In [ ]:
# Expanded Ecuador 2026 football metrics from Opta/StatsPerform JSON files.
# One-cell notebook source. Reads directly from /Users/marclamberts/Event data/Ecuador 2026.

from pathlib import Path
from datetime import datetime
import json
import math
import re
import shutil
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

SOURCE_DIR = Path('/Users/marclamberts/Event data/Ecuador 2026')
OUT_DIR = SOURCE_DIR / 'Aggregated'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEASON = 'Ecuador 2026'
TODAY = datetime.now().strftime('%B %-d, %Y')

SHOT_TYPES = {13, 14, 15, 16}
PASS_TYPES = {1, 2}
SUCCESS_PASS_TYPES = {1}
DEF_TYPES = {7, 8, 12, 44, 49, 50, 52, 54, 55, 61}
RECOVERY_TYPES = {49, 52}
TURNOVER_TYPES = {5, 6, 50}
SET_PIECE_QIDS = {2, 5, 6, 107, 124}
CROSS_QIDS = {2}
CORNER_QIDS = {6}
FREE_KICK_QIDS = {5}
THROW_IN_QIDS = {107}
Q_END_X, Q_END_Y = 140, 141

TYPE_LABELS = {
    1: 'pass', 2: 'offside_pass', 3: 'take_on', 4: 'foul', 5: 'out',
    6: 'corner_awarded', 7: 'tackle', 8: 'interception', 10: 'save',
    11: 'claim', 12: 'clearance', 13: 'miss', 14: 'post', 15: 'saved_shot',
    16: 'goal', 17: 'card', 18: 'sub_off', 19: 'sub_on', 27: 'start_delay',
    28: 'end_delay', 30: 'end', 32: 'start', 34: 'lineup', 37: 'collection_end',
    40: 'formation_change', 41: 'punch', 42: 'good_skill', 43: 'deleted_event',
    44: 'aerial', 45: 'challenge', 49: 'ball_recovery', 50: 'dispossessed',
    51: 'error', 52: 'keeper_pick_up', 54: 'smother', 55: 'offside_provoked',
    56: 'shield_ball_oop', 57: 'foul_throw_in', 58: 'penalty_faced',
    59: 'keeper_sweeper', 60: 'chance_missed', 61: 'ball_touch', 65: 'contentious_referee_decision',
    68: 'referee_drop_ball', 70: 'injury_time_announcement', 71: 'coach_setup',
    74: 'blocked_pass', 79: 'coverage_interruption', 80: 'drop_of_ball',
    81: 'obstacle', 83: 'attempted_tackle', 84: 'deleted_after_review',
    90: 'var_started', 91: 'var_outcome',
}

def safe_float(v, default=np.nan):
    try:
        if v is None or v == '':
            return default
        return float(v)
    except Exception:
        return default

def safe_int(v, default=0):
    try:
        if v is None or v == '':
            return default
        return int(float(v))
    except Exception:
        return default

def qdict(event):
    out = {}
    for q in event.get('qualifier', []) or []:
        qid = q.get('qualifierId')
        out[qid] = q.get('value', True)
    return out

def split_values(value):
    if value is True or value is None:
        return []
    return [x.strip() for x in str(value).split(',') if x.strip()]

def parse_match_name(path):
    stem = path.stem
    m = re.match(r'(\d{4}-\d{2}-\d{2})_(.+?) - (.+)$', stem)
    if not m:
        return '', stem, '', ''
    return m.group(1), f'{m.group(2)} - {m.group(3)}', m.group(2), m.group(3)

def minute_value(event):
    return safe_float(event.get('timeMin'), 0.0) + safe_float(event.get('timeSec'), 0.0) / 60.0

def match_length(details, events):
    ml = safe_float(details.get('matchLengthMin'), np.nan) + safe_float(details.get('matchLengthSec'), 0.0) / 60.0
    if not np.isfinite(ml) or ml <= 0:
        ml = max([minute_value(e) for e in events if e.get('periodId') in {1, 2, 3, 4}] or [90.0])
    return max(90.0, ml)

def shot_xg(x, y, q):
    if 9 in q:
        return 0.76
    x = safe_float(x, 0.0)
    y = safe_float(y, 50.0)
    dx = max(0.1, 100.0 - x)
    dy = abs(50.0 - y)
    dist = math.sqrt(dx * dx + dy * dy)
    angle = math.atan2(7.32 * dx, dx * dx + dy * dy - (7.32 / 2) ** 2)
    z = -2.9 - 0.075 * dist + 1.35 * angle
    base = 1 / (1 + math.exp(-z))
    if x >= 88 and 21.1 <= y <= 78.9:
        base *= 1.25
    if 26 in q or 72 in q:
        base *= 1.18
    if 28 in q or 82 in q:
        base *= 0.75
    return float(max(0.01, min(0.76, base)))

def pitch_value(x, y):
    x = safe_float(x, 0.0)
    y = safe_float(y, 50.0)
    dx = max(0.1, 100.0 - x)
    dy = abs(50.0 - y)
    central = max(0.0, 1.0 - dy / 50.0)
    box = 1.0 if x >= 83 and 21 <= y <= 79 else 0.0
    z = -5.0 + 0.065 * x + 0.9 * central + 0.6 * box - 0.012 * dx
    return float(max(0.0, min(0.65, 1 / (1 + math.exp(-z)))))

def expected_pass_completion(x, y, end_x, end_y, distance, dx, is_cross=False, set_piece=False):
    x = safe_float(x, 50.0)
    y = safe_float(y, 50.0)
    end_x = safe_float(end_x, x)
    end_y = safe_float(end_y, y)
    distance = safe_float(distance, 0.0)
    angle_change = abs(end_y - y) / 100.0
    forward = max(0.0, dx)
    backward = max(0.0, -dx)
    central_end = max(0.0, 1.0 - abs(50.0 - end_y) / 50.0)
    box_end = 1.0 if end_x >= 83 and 21 <= end_y <= 79 else 0.0
    own_third_start = 1.0 if x < 33.3 else 0.0
    z = (
        2.75
        - 0.045 * distance
        - 0.018 * forward
        + 0.010 * backward
        - 1.10 * angle_change
        - 0.90 * box_end
        - 0.55 * int(is_cross)
        - 0.20 * int(set_piece)
        + 0.25 * central_end
        + 0.20 * own_third_start
    )
    return float(max(0.03, min(0.98, 1 / (1 + math.exp(-z)))))

def zone_name(x, y):
    x = safe_float(x, 0.0)
    y = safe_float(y, 50.0)
    third = 'defensive' if x < 33.3 else 'middle' if x < 66.7 else 'attacking'
    lane = 'left' if y < 33.3 else 'central' if y < 66.7 else 'right'
    if x >= 83 and 21 <= y <= 79:
        return 'box'
    if x >= 66.7 and 21 <= y <= 79:
        return 'central_final_third'
    return f'{third}_{lane}'

def per(value, denom):
    value = safe_float(value, 0.0)
    denom = safe_float(denom, 0.0)
    return np.nan if denom == 0 else value / denom

def pct_rank(s):
    return s.rank(pct=True).fillna(0.0) * 100

json_files = sorted([p for p in SOURCE_DIR.glob('*.json') if p.is_file()])
if not json_files:
    raise FileNotFoundError(f'No JSON files found in {SOURCE_DIR}')

events_rows = []
minutes_rows = []
match_rows = []
player_names = {}
team_names = {}

for path in json_files:
    try:
        data = json.loads(path.read_text())
    except Exception as exc:
        print(f'Skipping unreadable JSON: {path.name} ({exc})')
        continue

    events = data.get('event', []) or data.get('events', []) or data.get('liveData', {}).get('event', [])
    details = data.get('matchDetails', {}) or {}
    match_date, match_name, home_name, away_name = parse_match_name(path)
    mlen = match_length(details, events)

    lineup_events = [e for e in events if e.get('typeId') == 34]
    team_order = []
    for e in lineup_events:
        cid = e.get('contestantId')
        if cid and cid not in team_order:
            team_order.append(cid)
    if len(team_order) >= 1:
        team_names[team_order[0]] = home_name or team_order[0]
    if len(team_order) >= 2:
        team_names[team_order[1]] = away_name or team_order[1]

    scores = details.get('scores', {}).get('total', {}) or {}
    home_goals = safe_int(scores.get('home'), 0)
    away_goals = safe_int(scores.get('away'), 0)
    team_goal_map = {}
    if len(team_order) >= 2:
        team_goal_map[team_order[0]] = {'gf': home_goals, 'ga': away_goals}
        team_goal_map[team_order[1]] = {'gf': away_goals, 'ga': home_goals}

    starters_by_team = defaultdict(list)
    roster_by_team = defaultdict(list)
    position_group = {}
    shirt_no = {}
    for e in lineup_events:
        cid = e.get('contestantId')
        q = qdict(e)
        ids = split_values(q.get(30))
        roles = split_values(q.get(44))
        shirts = split_values(q.get(59))
        lineup_slots = split_values(q.get(131))
        roster_by_team[cid].extend(ids)
        for idx, pid in enumerate(ids):
            if idx < len(shirts):
                shirt_no[pid] = shirts[idx]
            role_code = roles[idx] if idx < len(roles) else ''
            position_group[pid] = {'1': 'GK', '2': 'DEF', '3': 'MID', '4': 'FWD', '5': 'SUB'}.get(str(role_code), role_code)
            is_start = idx < 11
            if idx < len(lineup_slots):
                is_start = safe_int(lineup_slots[idx], 0) > 0
            if is_start:
                starters_by_team[cid].append(pid)

    intervals = defaultdict(lambda: [None, None])
    for cid, starters in starters_by_team.items():
        for pid in starters:
            intervals[(cid, pid)] = [0.0, mlen]
    for e in events:
        tid = e.get('typeId')
        if tid not in {18, 19} or not e.get('playerId'):
            continue
        cid, pid, tm = e.get('contestantId'), e.get('playerId'), min(mlen, minute_value(e))
        player_names[pid] = e.get('playerName') or player_names.get(pid, pid)
        if tid == 18:
            start, _ = intervals[(cid, pid)]
            intervals[(cid, pid)] = [0.0 if start is None else start, tm]
        elif tid == 19:
            _, end = intervals[(cid, pid)]
            intervals[(cid, pid)] = [tm, mlen if end is None else end]

    goal_events = []
    for e in events:
        if e.get('typeId') == 16:
            goal_events.append((minute_value(e), e.get('contestantId')))

    for (cid, pid), (start, end) in intervals.items():
        if start is None:
            start = 0.0
        if end is None:
            end = mlen
        mins = max(0.0, min(mlen, end) - max(0.0, start))
        if mins <= 0:
            continue
        gf_on = sum(1 for tm, gcid in goal_events if gcid == cid and start <= tm <= end)
        ga_on = sum(1 for tm, gcid in goal_events if gcid != cid and start <= tm <= end)
        minutes_rows.append({
            'season': SEASON, 'match_file': path.name, 'date': match_date, 'match': match_name,
            'team_id': cid, 'team': team_names.get(cid, cid), 'player_id': pid,
            'player': player_names.get(pid, pid), 'position_group': position_group.get(pid, ''),
            'shirt_no': shirt_no.get(pid, ''), 'minutes': mins, 'start_minute': start, 'end_minute': end,
            'started': int(start <= 0.01), 'sub_on': int(start > 0.01), 'sub_off': int(end < mlen - 0.01),
            'gf_on': gf_on, 'ga_on': ga_on, 'gd_on': gf_on - ga_on,
        })

    action_events = [e for e in events if e.get('periodId') in {1, 2, 3, 4} and e.get('playerId')]
    action_events.sort(key=lambda e: (safe_int(e.get('periodId'), 0), minute_value(e), safe_int(e.get('eventId'), 0)))
    prev_by_team = {}
    prev_by_player = {}
    corner_context = {}

    for e in action_events:
        tid = safe_int(e.get('typeId'), -1)
        q = qdict(e)
        cid = e.get('contestantId')
        pid = e.get('playerId')
        pname = e.get('playerName') or pid
        player_names[pid] = pname

        x = safe_float(e.get('x'), np.nan)
        y = safe_float(e.get('y'), np.nan)
        end_x = safe_float(q.get(Q_END_X), np.nan)
        end_y = safe_float(q.get(Q_END_Y), np.nan)
        if not np.isfinite(end_x):
            end_x = x
        if not np.isfinite(end_y):
            end_y = y
        dx = end_x - x if np.isfinite(x) and np.isfinite(end_x) else 0.0
        dy = end_y - y if np.isfinite(y) and np.isfinite(end_y) else 0.0
        distance = math.sqrt(dx * dx + dy * dy)
        success = int(safe_int(e.get('outcome'), 0) == 1)
        is_pass = tid in PASS_TYPES
        is_success_pass = tid in SUCCESS_PASS_TYPES and success == 1
        is_shot = tid in SHOT_TYPES
        is_goal = tid == 16
        xg = shot_xg(x, y, q) if is_shot else 0.0
        start_val = pitch_value(x, y) if np.isfinite(x) and np.isfinite(y) else 0.0
        end_val = pitch_value(end_x, end_y) if np.isfinite(end_x) and np.isfinite(end_y) else start_val
        xt = end_val - start_val if (is_success_pass and np.isfinite(end_x) and np.isfinite(end_y)) else 0.0
        danger = (xg if is_shot else max(0.0, xt) * 0.35 + (0.015 if x >= 83 and 21 <= y <= 79 else 0.0))
        progressive = int(is_success_pass and (dx >= 10 or (x < 50 and end_x >= 66.7) or (x < 66.7 and end_x >= 83)))
        final3_entry = int(is_success_pass and x < 66.7 and end_x >= 66.7)
        box_entry = int(is_success_pass and not (x >= 83 and 21 <= y <= 79) and end_x >= 83 and 21 <= end_y <= 79)
        penalty_box_touch = int(x >= 83 and 21 <= y <= 79)
        central_touch = int(21 <= y <= 79)
        halfspace_touch = int(21 <= y < 33 or 67 < y <= 79)
        wide_touch = int(y < 21 or y > 79)
        deep_touch = int(x < 33.3)
        final3_touch = int(x >= 66.7)
        high_value_touch = int(pitch_value(x, y) >= 0.12)
        set_piece = int(bool(set(q).intersection(SET_PIECE_QIDS)))
        prev_team = prev_by_team.get(cid)
        is_wide_start = bool(np.isfinite(y) and (y <= 21 or y >= 79))
        is_box_end = bool(np.isfinite(end_x) and np.isfinite(end_y) and end_x >= 83 and 21 <= end_y <= 79)
        location_cross = bool(is_pass and x >= 55 and is_wide_start and is_box_end and abs(end_y - y) >= 12)
        is_cross = bool(is_pass and (bool(set(q).intersection(CROSS_QIDS)) or location_cross))
        accurate_cross = int(is_cross and success)
        corner = int(is_pass and bool(set(q).intersection(CORNER_QIDS)))
        free_kick = int(is_pass and bool(set(q).intersection(FREE_KICK_QIDS)))
        throw_in = int(is_pass and bool(set(q).intersection(THROW_IN_QIDS)))
        recent_corner = corner_context.get(cid)
        corner_elapsed = minute_value(e) - recent_corner['minute'] if recent_corner else np.nan
        corner_phase_active = bool(recent_corner and 0 <= corner_elapsed <= 0.75)
        corner_first_phase = int(corner or (corner_phase_active and corner_elapsed <= 0.20))
        corner_second_phase = int(corner_phase_active and 0.20 < corner_elapsed <= 0.75)
        early_cross = int(is_cross and x < 80)
        deep_cross = int(is_cross and x >= 80)
        cutback = int(is_cross and x >= 83 and is_wide_start and end_x >= 83 and 33 <= end_y <= 67)
        six_yard_cross = int(is_cross and end_x >= 94 and 36 <= end_y <= 64)
        left_flank_cross = int(is_cross and y <= 21)
        right_flank_cross = int(is_cross and y >= 79)
        accurate_left_flank_cross = int(left_flank_cross and success)
        accurate_right_flank_cross = int(right_flank_cross and success)
        cross_xt = max(0.0, end_val - start_val) if is_cross else 0.0
        cross_threat = (cross_xt + (0.04 if is_box_end else 0.0) + (0.03 if cutback else 0.0)) if is_cross else 0.0
        set_piece_xt = max(0.0, end_val - start_val) if set_piece and is_pass else 0.0
        set_piece_shot = int(set_piece and is_shot)
        set_piece_xg = xg if set_piece and is_shot else 0.0
        set_piece_goal = int(set_piece and is_goal)
        penalty_shot = int(is_shot and 9 in q)
        penalty_goal = int(is_goal and 9 in q)
        non_penalty_goal = int(is_goal and 9 not in q)
        head_shot = int(is_shot and 15 in q)
        head_goal = int(is_goal and 15 in q)
        direct_free_kick = int(is_shot and free_kick)
        direct_free_kick_on_target = int(direct_free_kick and tid in {15, 16})
        corner_shot = int((corner_first_phase or corner_second_phase) and is_shot)
        corner_xg = xg if corner_shot else 0.0
        corner_goal = int((corner_first_phase or corner_second_phase) and is_goal)
        corner_first_phase_shot = int(corner_first_phase and is_shot)
        corner_first_phase_xg = xg if corner_first_phase_shot else 0.0
        corner_first_phase_goal = int(corner_first_phase and is_goal)
        corner_second_phase_shot = int(corner_second_phase and is_shot)
        corner_second_phase_xg = xg if corner_second_phase_shot else 0.0
        corner_second_phase_goal = int(corner_second_phase and is_goal)
        def_action = int(tid in DEF_TYPES)
        defensive_duel = int(tid in {7, 44, 45, 83})
        defensive_duel_won = int(defensive_duel and success)
        ground_duel = int(tid in {7, 45, 83})
        ground_duel_won = int(ground_duel and success)
        defensive_third_def_action = int(def_action and x < 33.3)
        middle_third_def_action = int(def_action and 33.3 <= x < 66.7)
        final_third_def_action = int(def_action and x >= 66.7)
        high_regain = int(tid in RECOVERY_TYPES and x >= 66.7)
        counterpress_regain = int(bool(prev_team) and tid in RECOVERY_TYPES and x >= 50 and prev_team.get('contestantId') != cid and minute_value(e) - prev_team.get('minute', 0) <= 0.15)
        last_line_action = int(def_action and x < 20)
        box_def_action = int(def_action and x <= 17 and 21 <= y <= 79)
        xpass = expected_pass_completion(x, y, end_x, end_y, distance, dx, is_cross=is_cross, set_piece=bool(set_piece)) if is_pass else 0.0
        xpass_completed = xpass if is_pass else 0.0
        pass_completion_oe = (success - xpass) if is_pass else 0.0
        pass_value_over_expected = pass_completion_oe * max(0.0, end_val - start_val) if is_pass else 0.0
        smart_pass = int(is_pass and end_x >= 70 and 21 <= end_y <= 79 and dx >= 10 and xpass <= 0.55)
        through_pass = int(is_pass and end_x >= 66.7 and dx >= 12 and 33 <= end_y <= 67)
        short_medium_pass = int(is_pass and distance < 30)
        accurate_forward_pass = int(is_success_pass and dx > 5)
        accurate_back_pass = int(is_success_pass and dx < -5)
        accurate_lateral_pass = int(is_success_pass and abs(dx) <= 5)
        accurate_long_pass = int(is_success_pass and distance >= 30)
        accurate_short_medium_pass = int(is_success_pass and distance < 30)
        disruption_zone_value = pitch_value(100 - x, y) if np.isfinite(x) and np.isfinite(y) else 0.0
        xdisruption = 0.0
        if def_action:
            xdisruption = (
                0.015
                + disruption_zone_value * 0.35
                + 0.020 * success
                + 0.030 * high_regain
                + 0.035 * counterpress_regain
                + 0.020 * box_def_action
                + 0.012 * defensive_duel_won
                + 0.010 * int(tid in {8, 49, 52})
            )
        xdisruption = float(max(0.0, xdisruption))

        prev_player = prev_by_player.get(pid)
        carry = 0
        carry_xt = 0.0
        carry_distance = 0.0
        carry_progressive = 0
        if prev_player and np.isfinite(prev_player['x']) and np.isfinite(prev_player['y']) and np.isfinite(x) and np.isfinite(y):
            dt = minute_value(e) - prev_player['minute']
            cd = math.sqrt((x - prev_player['x']) ** 2 + (y - prev_player['y']) ** 2)
            if 0.02 <= dt <= 0.35 and 2 <= cd <= 45:
                carry = 1
                carry_distance = cd
                carry_xt = pitch_value(x, y) - pitch_value(prev_player['x'], prev_player['y'])
                carry_progressive = int((x - prev_player['x']) >= 10 or (prev_player['x'] < 66.7 and x >= 83))

        chain_xt = xt + carry_xt
        possession_kept = int(success == 1 and tid not in TURNOVER_TYPES)
        pressure_regain = int(bool(prev_team) and tid in RECOVERY_TYPES and prev_team.get('contestantId') != cid and minute_value(e) - prev_team.get('minute', 0) <= 0.15)
        counter_threat = int(bool(prev_team) and prev_team.get('contestantId') != cid and minute_value(e) - prev_team.get('minute', 0) <= 0.25 and max(xt, carry_xt, 0) > 0.01)

        events_rows.append({
            'season': SEASON, 'match_file': path.name, 'date': match_date, 'match': match_name,
            'team_id': cid, 'team': team_names.get(cid, cid), 'player_id': pid, 'player': pname,
            'minute': minute_value(e), 'period': e.get('periodId'), 'type_id': tid, 'type': TYPE_LABELS.get(tid, f'type_{tid}'),
            'outcome': success, 'x': x, 'y': y, 'end_x': end_x, 'end_y': end_y,
            'zone': zone_name(x, y), 'end_zone': zone_name(end_x, end_y),
            'actions': 1, 'touches': int(tid not in {18, 19, 30, 32, 34, 37, 40, 70, 71, 90, 91}),
            'passes': int(is_pass), 'completed_passes': int(is_success_pass), 'incomplete_passes': int(is_pass and not is_success_pass),
            'xpass': xpass_completed, 'pass_completion_oe': pass_completion_oe,
            'pass_value_over_expected': pass_value_over_expected,
            'forward_passes': int(is_pass and dx > 5), 'backward_passes': int(is_pass and dx < -5), 'lateral_passes': int(is_pass and abs(dx) <= 5),
            'accurate_forward_passes': accurate_forward_pass, 'accurate_back_passes': accurate_back_pass,
            'accurate_lateral_passes': accurate_lateral_pass, 'long_passes': int(is_pass and distance >= 30),
            'accurate_long_passes': accurate_long_pass, 'short_medium_passes': short_medium_pass,
            'accurate_short_medium_passes': accurate_short_medium_pass, 'avg_pass_length_sum': distance if is_pass else 0.0,
            'avg_long_pass_length_sum': distance if is_pass and distance >= 30 else 0.0,
            'smart_passes': smart_pass, 'accurate_smart_passes': int(smart_pass and success),
            'through_passes': through_pass, 'accurate_through_passes': int(through_pass and success),
            'progressive_passes': progressive,
            'final3_entries': final3_entry, 'box_entries': box_entry, 'successful_box_entries': box_entry,
            'passes_into_box': box_entry, 'passes_into_final3': final3_entry,
            'set_piece_actions': set_piece, 'open_play_actions': int(not set_piece),
            'crosses': int(is_cross), 'accurate_crosses': accurate_cross, 'open_play_crosses': int(is_cross and not set_piece),
            'set_piece_crosses': int(is_cross and set_piece), 'corner_crosses': int(is_cross and corner),
            'free_kick_crosses': int(is_cross and free_kick), 'early_crosses': early_cross,
            'deep_crosses': deep_cross, 'cutbacks': cutback, 'six_yard_crosses': six_yard_cross,
            'crosses_into_box': int(is_cross and is_box_end), 'cross_xt': cross_xt, 'cross_threat': cross_threat,
            'left_flank_crosses': left_flank_cross, 'accurate_left_flank_crosses': accurate_left_flank_cross,
            'right_flank_crosses': right_flank_cross, 'accurate_right_flank_crosses': accurate_right_flank_cross,
            'crosses_to_goalie_box': six_yard_cross,
            'corners': corner, 'free_kicks': free_kick, 'throw_ins': throw_in,
            'set_piece_passes': int(set_piece and is_pass), 'completed_set_piece_passes': int(set_piece and is_success_pass),
            'set_piece_box_entries': int(set_piece and box_entry), 'set_piece_shots': set_piece_shot,
            'set_piece_xg': set_piece_xg, 'set_piece_goals': set_piece_goal, 'set_piece_xt': set_piece_xt,
            'corner_phase_actions': int(corner_first_phase or corner_second_phase),
            'corner_first_phase_actions': corner_first_phase,
            'corner_second_phase_actions': corner_second_phase,
            'corner_shots': corner_shot, 'corner_xg': corner_xg, 'corner_goals': corner_goal,
            'corner_first_phase_shots': corner_first_phase_shot,
            'corner_first_phase_xg': corner_first_phase_xg,
            'corner_first_phase_goals': corner_first_phase_goal,
            'corner_second_phase_shots': corner_second_phase_shot,
            'corner_second_phase_xg': corner_second_phase_xg,
            'corner_second_phase_goals': corner_second_phase_goal,
            'take_ons': int(tid == 3), 'successful_take_ons': int(tid == 3 and success), 'failed_take_ons': int(tid == 3 and not success),
            'offensive_duels': int(tid == 3), 'offensive_duels_won': int(tid == 3 and success),
            'carries': carry, 'carry_distance': carry_distance, 'progressive_carries': carry_progressive,
            'progressive_runs': carry_progressive, 'accelerations': int(carry and carry_distance >= 12),
            'shots': int(is_shot), 'goals': int(is_goal), 'non_penalty_goals': non_penalty_goal,
            'penalty_shots': penalty_shot, 'penalty_goals': penalty_goal, 'head_shots': head_shot, 'head_goals': head_goal,
            'direct_free_kicks': direct_free_kick, 'direct_free_kicks_on_target': direct_free_kick_on_target,
            'on_target_shots': int(tid in {15, 16}),
            'blocked_or_saved_shots': int(tid == 15), 'post_shots': int(tid == 14), 'missed_shots': int(tid == 13),
            'xg': xg, 'psxg_proxy': xg * (1.20 if tid == 16 else 1.05 if tid == 15 else 0.25 if tid == 14 else 0.0),
            'danger': danger, 'xt': xt, 'xt_positive': max(0.0, xt), 'xt_negative': min(0.0, xt),
            'carry_xt': carry_xt, 'carry_xt_positive': max(0.0, carry_xt), 'chain_xt': chain_xt,
            'progression_distance': max(0.0, dx), 'pass_distance': distance if is_pass else 0.0,
            'def_actions': def_action, 'tackles': int(tid in {7, 83}), 'tackles_won': int(tid == 7 and success),
            'interceptions': int(tid == 8), 'recoveries': int(tid in RECOVERY_TYPES), 'clearances': int(tid == 12),
            'aerials': int(tid == 44), 'aerials_won': int(tid == 44 and success), 'blocks': int(tid in {74, 15}),
            'defensive_duels': defensive_duel, 'defensive_duels_won': defensive_duel_won,
            'ground_duels': ground_duel, 'ground_duels_won': ground_duel_won,
            'defensive_third_def_actions': defensive_third_def_action,
            'middle_third_def_actions': middle_third_def_action,
            'final_third_def_actions': final_third_def_action,
            'high_regains': high_regain, 'counterpress_regains': counterpress_regain,
            'last_line_actions': last_line_action, 'box_def_actions': box_def_action,
            'xdisruption': xdisruption,
            'saves': int(tid == 10), 'claims': int(tid == 11), 'keeper_pickups': int(tid == 52),
            'smothers': int(tid == 54), 'keeper_sweeper_actions': int(tid == 59), 'punches': int(tid == 41),
            'penalties_faced': int(tid == 58), 'gk_actions': int(tid in {10, 11, 41, 52, 54, 58, 59}),
            'fouls': int(tid == 4 and success), 'fouled': int(tid == 4 and not success), 'cards': int(tid == 17),
            'losses': int(tid in TURNOVER_TYPES or (is_pass and not is_success_pass)), 'dispossessed': int(tid == 50),
            'possession_kept': possession_kept, 'pressure_regains': pressure_regain, 'counter_threat_actions': counter_threat,
            'box_touches': penalty_box_touch, 'final3_touches': final3_touch, 'deep_touches': deep_touch,
            'central_touches': central_touch, 'halfspace_touches': halfspace_touch, 'wide_touches': wide_touch,
            'high_value_touches': high_value_touch,
            'territory_added': dx if is_pass or carry else 0.0,
            'field_tilt_touch': int(x >= 66.7), 'own_third_security_touch': int(x < 33.3 and possession_kept),
        })

        if corner:
            corner_context[cid] = {'minute': minute_value(e), 'event_id': e.get('id')}
        prev_by_team[cid] = {'contestantId': cid, 'minute': minute_value(e), 'x': x, 'y': y}
        prev_by_player[pid] = {'minute': minute_value(e), 'x': end_x if np.isfinite(end_x) else x, 'y': end_y if np.isfinite(end_y) else y}

    match_rows.append({
        'season': SEASON, 'match_file': path.name, 'date': match_date, 'match': match_name,
        'home_team': home_name, 'away_team': away_name, 'home_goals': home_goals, 'away_goals': away_goals,
        'match_length': mlen, 'events': len(events),
    })

events_df = pd.DataFrame(events_rows)
minutes_df = pd.DataFrame(minutes_rows)
matches_df = pd.DataFrame(match_rows)

if events_df.empty:
    raise RuntimeError('No player events found.')

for col in [
    'assists', 'xa', 'shot_assists', 'key_passes', 'second_assists', 'third_assists',
    'received_passes', 'received_long_passes', 'deep_completions', 'deep_completed_crosses',
    'successful_attacking_actions'
]:
    events_df[col] = 0.0

events_df = events_df.sort_values(['match_file', 'period', 'minute']).reset_index(drop=True)
for (_, _), g in events_df.groupby(['match_file', 'period'], sort=False):
    idxs = list(g.index)
    for pos, idx in enumerate(idxs):
        row = events_df.loc[idx]
        if row.get('passes', 0) and row.get('outcome', 0) == 1:
            if row.get('end_x', 0) >= 83 and 21 <= row.get('end_y', 50) <= 79:
                events_df.at[idx, 'deep_completions'] = 1
                if row.get('crosses', 0):
                    events_df.at[idx, 'deep_completed_crosses'] = 1
            for j in idxs[pos + 1: min(pos + 8, len(idxs))]:
                nxt = events_df.loc[j]
                if nxt['minute'] - row['minute'] > 0.18:
                    break
                if nxt['team_id'] == row['team_id'] and nxt['player_id'] != row['player_id'] and nxt.get('touches', 0):
                    events_df.at[j, 'received_passes'] += 1
                    if row.get('pass_distance', 0) >= 30:
                        events_df.at[j, 'received_long_passes'] += 1
                    break
        if row.get('shots', 0):
            same_team_prev = []
            for j in reversed(idxs[max(0, pos - 12):pos]):
                prv = events_df.loc[j]
                if row['minute'] - prv['minute'] > 0.25:
                    break
                if prv['team_id'] == row['team_id'] and prv.get('passes', 0):
                    same_team_prev.append(j)
            if same_team_prev:
                assist_idx = same_team_prev[0]
                events_df.at[assist_idx, 'shot_assists'] += 1
                events_df.at[assist_idx, 'key_passes'] += 1
                events_df.at[assist_idx, 'xa'] += row.get('xg', 0)
                if row.get('goals', 0):
                    events_df.at[assist_idx, 'assists'] += 1
                if len(same_team_prev) > 1:
                    events_df.at[same_team_prev[1], 'second_assists'] += 1
                if len(same_team_prev) > 2:
                    events_df.at[same_team_prev[2], 'third_assists'] += 1

events_df['successful_attacking_actions'] = (
    events_df.get('successful_take_ons', 0)
    + events_df.get('shots', 0)
    + events_df.get('shot_assists', 0)
    + events_df.get('deep_completions', 0)
)

metric_cols = [c for c in events_df.columns if c not in {
    'season', 'match_file', 'date', 'match', 'team_id', 'team', 'player_id', 'player',
    'minute', 'period', 'type_id', 'type', 'outcome', 'x', 'y', 'end_x', 'end_y', 'zone', 'end_zone'
}]
player_match_actions = events_df.groupby(
    ['season', 'match_file', 'date', 'match', 'team_id', 'team', 'player_id', 'player'],
    as_index=False
)[metric_cols].sum()

if minutes_df.empty:
    minutes_df = player_match_actions[['season', 'match_file', 'date', 'match', 'team_id', 'team', 'player_id', 'player']].copy()
    minutes_df['minutes'] = np.nan
    minutes_df[['gf_on', 'ga_on', 'gd_on', 'started', 'sub_on', 'sub_off']] = 0

player_match = minutes_df.merge(
    player_match_actions,
    on=['season', 'match_file', 'date', 'match', 'team_id', 'team', 'player_id', 'player'],
    how='outer'
)
for c in metric_cols + ['minutes', 'gf_on', 'ga_on', 'gd_on', 'started', 'sub_on', 'sub_off']:
    if c in player_match:
        player_match[c] = player_match[c].fillna(0)

team_event_against = events_df.groupby(['match_file', 'team_id'], as_index=False).agg(
    team_shots=('shots', 'sum'),
    team_shots_on_target=('on_target_shots', 'sum'),
    team_xg=('xg', 'sum'),
    team_goals=('goals', 'sum'),
)
opp_rows = []
for match_file, g in team_event_against.groupby('match_file'):
    for _, row in g.iterrows():
        opp = g[g['team_id'] != row['team_id']]
        opp_rows.append({
            'match_file': match_file,
            'team_id': row['team_id'],
            'opp_shots_total': opp['team_shots'].sum(),
            'opp_shots_on_target_total': opp['team_shots_on_target'].sum(),
            'opp_xg_total': opp['team_xg'].sum(),
            'opp_goals_total': opp['team_goals'].sum(),
        })
opp_df = pd.DataFrame(opp_rows)
player_match = player_match.merge(opp_df, on=['match_file', 'team_id'], how='left')
match_lengths = matches_df.set_index('match_file')['match_length'].to_dict()
player_match['match_length'] = player_match['match_file'].map(match_lengths).fillna(90.0)
minute_share = (player_match['minutes'] / player_match['match_length'].replace(0, np.nan)).clip(lower=0, upper=1).fillna(0)
player_match['shots_against'] = player_match['opp_shots_total'].fillna(0) * minute_share
player_match['shots_on_target_against'] = player_match['opp_shots_on_target_total'].fillna(0) * minute_share
player_match['xg_against'] = player_match['opp_xg_total'].fillna(0) * minute_share
player_match['conceded_goals'] = player_match['ga_on']
player_match['clean_sheets'] = ((player_match['ga_on'] == 0) & (player_match['minutes'] >= 60)).astype(int)
player_match['save_rate'] = player_match.get('saves', 0) / player_match['shots_on_target_against'].replace(0, np.nan)
player_match['prevented_goals'] = player_match['xg_against'] - player_match['conceded_goals']

def add_rates(df, minutes_col='minutes'):
    mins = df[minutes_col].replace(0, np.nan)
    actions = df.get('actions', pd.Series(0, index=df.index)).replace(0, np.nan)
    passes = df.get('passes', pd.Series(0, index=df.index)).replace(0, np.nan)
    shots = df.get('shots', pd.Series(0, index=df.index)).replace(0, np.nan)
    for col in ['gda', 'action_gda', 'threat', 'xg', 'psxg_proxy', 'danger', 'xt', 'xt_positive', 'carry_xt',
                'chain_xt', 'passes', 'completed_passes', 'forward_passes', 'backward_passes',
                'lateral_passes', 'progressive_passes', 'final3_entries', 'box_entries',
                'carries', 'progressive_carries', 'shots', 'goals', 'def_actions', 'recoveries', 'tackles',
                'interceptions', 'clearances', 'touches', 'losses', 'pressure_regains', 'counter_threat_actions',
                'box_touches', 'final3_touches', 'high_value_touches', 'territory_added',
                'two_way_value', 'control_value', 'saves', 'claims', 'keeper_pickups', 'smothers',
                'keeper_sweeper_actions', 'punches', 'penalties_faced', 'gk_actions',
                'central_touches', 'halfspace_touches', 'wide_touches', 'deep_touches', 'field_tilt_touch',
                'take_ons', 'fouled', 'aerials', 'passes_into_box', 'passes_into_final3',
                'crosses', 'accurate_crosses', 'open_play_crosses',
                'set_piece_crosses', 'corner_crosses', 'free_kick_crosses', 'early_crosses',
                'deep_crosses', 'cutbacks', 'six_yard_crosses', 'crosses_into_box', 'cross_xt',
                'cross_threat', 'corners', 'free_kicks', 'throw_ins', 'set_piece_passes',
                'completed_set_piece_passes', 'set_piece_box_entries', 'set_piece_shots',
                'set_piece_xg', 'set_piece_goals', 'set_piece_xt', 'defensive_duels',
                'defensive_duels_won', 'ground_duels', 'ground_duels_won',
                'blocks', 'defensive_third_def_actions', 'middle_third_def_actions',
                'final_third_def_actions', 'high_regains', 'counterpress_regains',
                'last_line_actions', 'box_def_actions', 'xpass', 'pass_completion_oe',
                'pass_value_over_expected', 'xdisruption', 'corner_phase_actions',
                'corner_first_phase_actions', 'corner_second_phase_actions', 'corner_shots',
                'corner_xg', 'corner_goals', 'corner_first_phase_shots',
                'corner_first_phase_xg', 'corner_first_phase_goals',
                'corner_second_phase_shots', 'corner_second_phase_xg',
                'corner_second_phase_goals', 'assists', 'xa', 'shot_assists', 'key_passes',
                'second_assists', 'third_assists', 'received_passes', 'received_long_passes',
                'deep_completions', 'deep_completed_crosses', 'successful_attacking_actions',
                'non_penalty_goals', 'head_goals', 'penalty_shots', 'penalty_goals', 'fouls',
                'direct_free_kicks', 'direct_free_kicks_on_target', 'accurate_forward_passes',
                'accurate_back_passes', 'accurate_lateral_passes', 'accurate_long_passes',
                'short_medium_passes', 'accurate_short_medium_passes', 'smart_passes',
                'accurate_smart_passes', 'through_passes', 'accurate_through_passes',
                'left_flank_crosses', 'accurate_left_flank_crosses', 'right_flank_crosses',
                'accurate_right_flank_crosses', 'crosses_to_goalie_box', 'offensive_duels',
                'offensive_duels_won', 'progressive_runs', 'accelerations', 'conceded_goals',
                'shots_against', 'shots_on_target_against', 'xg_against', 'prevented_goals',
                'clean_sheets']:
        if col in df:
            df[f'{col}_p90'] = df[col] / mins * 90
    if 'completed_passes' in df and 'passes' in df:
        df['pass_pct'] = df['completed_passes'] / passes
    if 'forward_passes' in df:
        df['forward_pass_pct'] = df['forward_passes'] / passes
        df['accurate_forward_pass_pct'] = df.get('accurate_forward_passes', 0) / df['forward_passes'].replace(0, np.nan)
    if 'backward_passes' in df:
        df['accurate_back_pass_pct'] = df.get('accurate_back_passes', 0) / df['backward_passes'].replace(0, np.nan)
    if 'lateral_passes' in df:
        df['accurate_lateral_pass_pct'] = df.get('accurate_lateral_passes', 0) / df['lateral_passes'].replace(0, np.nan)
    if 'long_passes' in df:
        df['accurate_long_pass_pct'] = df.get('accurate_long_passes', 0) / df['long_passes'].replace(0, np.nan)
        df['avg_long_pass_length'] = df.get('avg_long_pass_length_sum', 0) / df['long_passes'].replace(0, np.nan)
    if 'short_medium_passes' in df:
        df['accurate_short_medium_pass_pct'] = df.get('accurate_short_medium_passes', 0) / df['short_medium_passes'].replace(0, np.nan)
    if 'avg_pass_length_sum' in df:
        df['avg_pass_length'] = df['avg_pass_length_sum'] / passes
    if 'smart_passes' in df:
        df['accurate_smart_pass_pct'] = df.get('accurate_smart_passes', 0) / df['smart_passes'].replace(0, np.nan)
    if 'through_passes' in df:
        df['accurate_through_pass_pct'] = df.get('accurate_through_passes', 0) / df['through_passes'].replace(0, np.nan)
    if 'progressive_passes' in df:
        df['progressive_pass_pct'] = df['progressive_passes'] / passes
    if 'goals' in df and 'xg' in df:
        df['goals_minus_xg'] = df['goals'] - df['xg']
        df['finishing_ratio'] = df['goals'] / df['xg'].replace(0, np.nan)
    if 'on_target_shots' in df:
        df['on_target_pct'] = df['on_target_shots'] / shots
    if 'xg' in df:
        df['xg_per_shot'] = df['xg'] / shots
    if 'xt' in df:
        df['xt_per_action'] = df['xt'] / actions
        df['xt_per100_actions'] = df['xt'] / actions * 100
        df['xt_per100_passes'] = df['xt'] / passes * 100
    if 'danger' in df:
        df['danger_per_action'] = df['danger'] / actions
    if 'losses' in df:
        df['security_pct'] = 1 - df['losses'] / actions
    if 'successful_take_ons' in df and 'take_ons' in df:
        df['take_on_pct'] = df['successful_take_ons'] / df['take_ons'].replace(0, np.nan)
    if 'aerials_won' in df and 'aerials' in df:
        df['aerial_win_pct'] = df['aerials_won'] / df['aerials'].replace(0, np.nan)
    if 'tackles_won' in df and 'tackles' in df:
        df['tackle_win_pct'] = df['tackles_won'] / df['tackles'].replace(0, np.nan)
    if 'territory_added' in df:
        df['territory_per_action'] = df['territory_added'] / actions
    if 'gd_on' in df:
        df['gd_on_p90'] = df['gd_on'] / mins * 90
    if 'touches' in df:
        touches = df['touches'].replace(0, np.nan)
        df['threat_per_touch'] = df.get('threat', 0) / touches
        df['xt_per_touch'] = df.get('xt', 0) / touches
        df['danger_per_touch'] = df.get('danger', 0) / touches
        df['box_touch_share'] = df.get('box_touches', 0) / touches
        df['final3_touch_share'] = df.get('final3_touches', 0) / touches
        df['central_touch_share'] = df.get('central_touches', 0) / touches
        df['wide_touch_share'] = df.get('wide_touches', 0) / touches
    if 'completed_passes' in df and 'progressive_passes' in df:
        df['progressive_pass_per_complete'] = df['progressive_passes'] / df['completed_passes'].replace(0, np.nan)
    if 'box_entries' in df and 'final3_entries' in df:
        df['box_entry_rate_from_final3'] = df['box_entries'] / df['final3_entries'].replace(0, np.nan)
    if 'xg' in df and 'box_touches' in df:
        df['xg_per_box_touch'] = df['xg'] / df['box_touches'].replace(0, np.nan)
    if 'xt_positive' in df and 'xt_negative' in df:
        df['xt_risk_reward'] = df['xt_positive'] / df['xt_negative'].abs().replace(0, np.nan)
    if 'progressive_carries' in df and 'progressive_passes' in df:
        df['carry_progression_share'] = df['progressive_carries'] / (df['progressive_carries'] + df['progressive_passes']).replace(0, np.nan)
        df['pass_progression_share'] = df['progressive_passes'] / (df['progressive_carries'] + df['progressive_passes']).replace(0, np.nan)
    if 'pressure_regains' in df and 'counter_threat_actions' in df:
        df['regain_to_threat_rate'] = df['counter_threat_actions'] / df['pressure_regains'].replace(0, np.nan)
    if 'def_actions' in df and 'actions' in df:
        df['def_action_share'] = df['def_actions'] / actions
    if 'crosses' in df:
        crosses = df['crosses'].replace(0, np.nan)
        df['cross_accuracy'] = df.get('accurate_crosses', 0) / crosses
        df['accurate_left_flank_cross_pct'] = df.get('accurate_left_flank_crosses', 0) / df.get('left_flank_crosses', 0).replace(0, np.nan)
        df['accurate_right_flank_cross_pct'] = df.get('accurate_right_flank_crosses', 0) / df.get('right_flank_crosses', 0).replace(0, np.nan)
        df['cross_threat_per_cross'] = df.get('cross_threat', 0) / crosses
        df['cross_xt_per_cross'] = df.get('cross_xt', 0) / crosses
        df['cutback_share'] = df.get('cutbacks', 0) / crosses
        df['open_play_cross_share'] = df.get('open_play_crosses', 0) / crosses
        df['set_piece_cross_share'] = df.get('set_piece_crosses', 0) / crosses
    if 'set_piece_passes' in df:
        sp_passes = df['set_piece_passes'].replace(0, np.nan)
        df['set_piece_pass_pct'] = df.get('completed_set_piece_passes', 0) / sp_passes
        df['set_piece_threat_per_action'] = (df.get('set_piece_xg', 0) + df.get('set_piece_xt', 0)) / df.get('set_piece_actions', 0).replace(0, np.nan)
    if 'defensive_duels' in df:
        df['defensive_duel_win_pct'] = df.get('defensive_duels_won', 0) / df['defensive_duels'].replace(0, np.nan)
    if 'offensive_duels' in df:
        df['offensive_duel_win_pct'] = df.get('offensive_duels_won', 0) / df['offensive_duels'].replace(0, np.nan)
    if 'ground_duels' in df:
        df['ground_duel_win_pct'] = df.get('ground_duels_won', 0) / df['ground_duels'].replace(0, np.nan)
    if 'defensive_third_def_actions' in df and 'def_actions' in df:
        defs = df['def_actions'].replace(0, np.nan)
        df['def_third_action_share'] = df['defensive_third_def_actions'] / defs
        df['mid_third_def_action_share'] = df.get('middle_third_def_actions', 0) / defs
        df['high_def_action_share'] = df.get('final_third_def_actions', 0) / defs
    if 'xpass' in df and 'passes' in df:
        df['xpass_completion_pct'] = df['xpass'] / df['passes'].replace(0, np.nan)
        df['pass_completion_above_expected'] = df.get('completed_passes', 0) - df['xpass']
        df['pass_completion_oe_per_pass'] = df.get('pass_completion_oe', 0) / df['passes'].replace(0, np.nan)
    if 'xdisruption' in df and 'def_actions' in df:
        df['xdisruption_per_def_action'] = df['xdisruption'] / df['def_actions'].replace(0, np.nan)
    if 'corners' in df:
        df['corner_xg_per_corner'] = df.get('corner_xg', 0) / df['corners'].replace(0, np.nan)
        df['corner_goal_rate'] = df.get('corner_goals', 0) / df['corners'].replace(0, np.nan)
        df['corner_second_phase_xg_share'] = df.get('corner_second_phase_xg', 0) / df.get('corner_xg', 0).replace(0, np.nan)
    if 'direct_free_kicks' in df:
        df['direct_free_kick_on_target_pct'] = df.get('direct_free_kicks_on_target', 0) / df['direct_free_kicks'].replace(0, np.nan)
    if 'saves' in df and 'shots_on_target_against' in df:
        df['save_rate'] = df['saves'] / df['shots_on_target_against'].replace(0, np.nan)
    return df

player_match['action_gda'] = (
    player_match.get('goals', 0) * 0.65 + player_match.get('xg', 0) * 0.35 +
    player_match.get('xt_positive', 0) * 0.55 + player_match.get('carry_xt_positive', 0) * 0.45 +
    player_match.get('def_actions', 0) * 0.008 + player_match.get('pressure_regains', 0) * 0.018 -
    player_match.get('losses', 0) * 0.006
)
player_match['threat'] = (
    player_match.get('danger', 0) + player_match.get('xt_positive', 0) + player_match.get('carry_xt_positive', 0) +
    player_match.get('xg', 0)
)
player_match['gda'] = player_match.get('gd_on', 0) + player_match['action_gda']
player_match['two_way_value'] = player_match['threat'] + player_match.get('def_actions', 0) * 0.012 + player_match.get('recoveries', 0) * 0.01 - player_match.get('losses', 0) * 0.005
player_match['control_value'] = player_match.get('completed_passes', 0) * 0.004 + player_match.get('security_pct', 0) * 0.2 + player_match.get('territory_added', 0) * 0.002
player_match = add_rates(player_match)

sum_cols = [c for c in player_match.select_dtypes(include=[np.number]).columns if c not in {'pass_pct', 'forward_pass_pct', 'progressive_pass_pct', 'on_target_pct', 'xg_per_shot', 'finishing_ratio', 'xt_per_action', 'xt_per100_actions', 'xt_per100_passes', 'danger_per_action', 'security_pct', 'take_on_pct', 'aerial_win_pct', 'tackle_win_pct', 'territory_per_action'}]
player_keys = ['season', 'team', 'player_id', 'player']
player_season = player_match.groupby(player_keys, as_index=False)[sum_cols].sum()
player_season['matches'] = player_match.groupby(player_keys)['match_file'].nunique().values
player_team_ids = player_match.groupby(player_keys)['team_id'].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0]).reset_index()
player_season = player_season.merge(player_team_ids, on=player_keys, how='left')
player_season = add_rates(player_season)

for df in [player_match, player_season]:
    mins = df['minutes'].replace(0, np.nan)
    df['availability_score'] = np.minimum(1.0, df['minutes'] / 900.0)
    df['reliability'] = np.sqrt(np.minimum(1.0, df['minutes'] / 900.0))
    df['gda_reliable_p90'] = df['gda_p90'] * df['reliability']
    df['threat_reliable_p90'] = df['threat_p90'] * df['reliability']
    df['impact_score_raw'] = (
        df['gda_p90'].fillna(0) * 0.30 + df['threat_p90'].fillna(0) * 0.25 +
        df['xt_p90'].fillna(0) * 0.20 + df['two_way_value_p90'].fillna(0) * 0.15 +
        df['control_value_p90'].fillna(0) * 0.10
    )
    df['attacking_index'] = df['threat_p90'].fillna(0) + df['xg_p90'].fillna(0) + df['box_entries_p90'].fillna(0) * 0.05
    df['progression_index'] = df['xt_p90'].fillna(0) + df['progressive_passes_p90'].fillna(0) * 0.04 + df['progressive_carries_p90'].fillna(0) * 0.05
    df['defensive_index'] = df['def_actions_p90'].fillna(0) * 0.08 + df['recoveries_p90'].fillna(0) * 0.06 + df['pressure_regains_p90'].fillna(0) * 0.10
    df['security_index'] = df['security_pct'].fillna(0) * 100 - df['losses_p90'].fillna(0)
    df['box_presence_index'] = df['box_touches_p90'].fillna(0) + df['shots_p90'].fillna(0) * 1.5 + df['xg_p90'].fillna(0) * 5
    df['transition_index'] = df['counter_threat_actions_p90'].fillna(0) * 0.6 + df['pressure_regains_p90'].fillna(0) * 0.4
    df['creative_engine_index'] = df['xt_per100_passes'].fillna(0) * 20 + df['progressive_pass_pct'].fillna(0) * 25 + df['final3_entries_p90'].fillna(0) * 0.4
    df['ball_winner_index'] = df['tackles_p90'].fillna(0) * 0.5 + df['interceptions_p90'].fillna(0) * 0.8 + df['recoveries_p90'].fillna(0) * 0.4
    df['finisher_index'] = df['xg_p90'].fillna(0) * 10 + df['goals_p90'].fillna(0) * 8 + df['on_target_pct'].fillna(0) * 2
    df['chance_quality_index'] = df['xg_per_shot'].fillna(0) * 100 + df['box_touch_share'].fillna(0) * 10 + df['on_target_pct'].fillna(0) * 5
    df['volume_quality_index'] = df['shots_p90'].fillna(0) * df['xg_per_shot'].fillna(0) * 10 + df['threat_p90'].fillna(0)
    df['needle_mover_index'] = df['gda_p90'].fillna(0) * 0.45 + df['threat_p90'].fillna(0) * 0.35 + df['xt_risk_reward'].clip(upper=5).fillna(0) * 0.20
    df['chaos_creator_index'] = df['take_ons_p90'].fillna(0) * 0.4 + df['box_touches_p90'].fillna(0) * 0.3 + df['counter_threat_actions_p90'].fillna(0) * 0.8 + df['fouled_p90'].fillna(0) * 0.25
    df['press_resistance_index'] = df['security_pct'].fillna(0) * 50 + df['take_on_pct'].fillna(0) * 20 + df['progressive_carries_p90'].fillna(0) * 0.4
    df['rest_defence_index'] = df['pressure_regains_p90'].fillna(0) * 0.9 + df['recoveries_p90'].fillna(0) * 0.4 + df['def_actions_p90'].fillna(0) * 0.1
    df['territory_domination_index'] = df['territory_added_p90'].fillna(0) * 0.015 + df['field_tilt_touch_p90'].fillna(0) * 0.05 + df['final3_entries_p90'].fillna(0) * 0.3
    df['connector_index'] = df['completed_passes_p90'].fillna(0) * 0.05 + df['pass_pct'].fillna(0) * 10 + df['central_touch_share'].fillna(0) * 5
    df['wide_progression_index'] = df['wide_touch_share'].fillna(0) * 10 + df['box_entries_p90'].fillna(0) * 0.6 + df['progressive_carries_p90'].fillna(0) * 0.25
    df['halfspace_value_index'] = df.get('halfspace_touches_p90', 0).fillna(0) * 0.08 + df['threat_per_touch'].fillna(0) * 20
    df['directness_index'] = df['forward_pass_pct'].fillna(0) * 20 + df['territory_per_action'].fillna(0) * 2 + df['progressive_pass_per_complete'].fillna(0) * 15
    df['vertical_threat_index'] = df['progressive_passes_p90'].fillna(0) * 0.3 + df['progressive_carries_p90'].fillna(0) * 0.4 + df['xt_p90'].fillna(0)
    df['defensive_range_index'] = df['interceptions_p90'].fillna(0) * 0.7 + df['tackles_p90'].fillna(0) * 0.4 + df['aerials_p90'].fillna(0) * 0.2
    df['goalkeeper_activity_index'] = df['gk_actions_p90'].fillna(0) * 0.7 + df['saves_p90'].fillna(0) * 0.8 + df['keeper_sweeper_actions_p90'].fillna(0) * 0.6
    df['crossing_value_index'] = df['cross_threat_p90'].fillna(0) * 2.0 + df['accurate_crosses_p90'].fillna(0) * 0.35 + df['cutbacks_p90'].fillna(0) * 0.6 + df['cross_accuracy'].fillna(0) * 2.0
    df['box_delivery_index'] = df['crosses_into_box_p90'].fillna(0) * 0.45 + df['six_yard_crosses_p90'].fillna(0) * 0.7 + df['cutbacks_p90'].fillna(0) * 0.65 + df['passes_into_box_p90'].fillna(0) * 0.25
    df['set_piece_creator_index'] = df['set_piece_xt_p90'].fillna(0) * 3.0 + df['set_piece_xg_p90'].fillna(0) * 4.0 + df['set_piece_box_entries_p90'].fillna(0) * 0.35 + df['set_piece_crosses_p90'].fillna(0) * 0.25
    df['dead_ball_threat_index'] = df['corners_p90'].fillna(0) * 0.15 + df['free_kicks_p90'].fillna(0) * 0.2 + df['set_piece_threat_per_action'].fillna(0) * 8.0
    df['corner_threat_index'] = df['corner_xg_p90'].fillna(0) * 5.0 + df['corner_second_phase_xg_p90'].fillna(0) * 6.0 + df['corner_shots_p90'].fillna(0) * 0.35 + df['corner_goal_rate'].fillna(0) * 2.0
    df['second_phase_corner_index'] = df['corner_second_phase_actions_p90'].fillna(0) * 0.25 + df['corner_second_phase_xg_p90'].fillna(0) * 8.0 + df['corner_second_phase_xg_share'].fillna(0) * 2.0
    df['defensive_duel_index'] = df['defensive_duels_p90'].fillna(0) * 0.3 + df['defensive_duel_win_pct'].fillna(0) * 3.0 + df['ground_duel_win_pct'].fillna(0) * 1.5
    df['high_press_defending_index'] = df['final_third_def_actions_p90'].fillna(0) * 0.45 + df['high_regains_p90'].fillna(0) * 0.8 + df['counterpress_regains_p90'].fillna(0) * 0.9
    df['box_defender_index'] = df['box_def_actions_p90'].fillna(0) * 0.8 + df['last_line_actions_p90'].fillna(0) * 0.5 + df['clearances_p90'].fillna(0) * 0.25 + df['blocks_p90'].fillna(0) * 0.35
    df['defensive_security_index'] = df['defensive_duel_win_pct'].fillna(0) * 2.0 + df['def_third_action_share'].fillna(0) * 1.5 + df['defensive_index'].fillna(0)
    df['disruption_index'] = df['xdisruption_p90'].fillna(0) * 3.0 + df['xdisruption_per_def_action'].fillna(0) * 8.0 + df['high_press_defending_index'].fillna(0) * 0.3
    df['passing_added_index'] = df['pass_completion_oe_per_pass'].fillna(0) * 20.0 + df['pass_value_over_expected_p90'].fillna(0) * 5.0 + df['progressive_passes_p90'].fillna(0) * 0.15
    df['waltzing_all_round_index'] = (
        df['attacking_index'].fillna(0) * 0.20 + df['progression_index'].fillna(0) * 0.20 +
        df['defensive_index'].fillna(0) * 0.20 + df['security_index'].fillna(0) * 0.15 +
        df['needle_mover_index'].fillna(0) * 0.25
    )
    df['gda_share_of_team'] = np.nan
    df['threat_share_of_team'] = np.nan

for col in ['impact_score_raw', 'gda_p90', 'threat_p90', 'xt_p90', 'attacking_index', 'progression_index', 'defensive_index', 'security_index', 'box_presence_index', 'transition_index', 'creative_engine_index', 'ball_winner_index', 'finisher_index', 'chance_quality_index', 'volume_quality_index', 'needle_mover_index', 'chaos_creator_index', 'press_resistance_index', 'rest_defence_index', 'territory_domination_index', 'connector_index', 'wide_progression_index', 'halfspace_value_index', 'directness_index', 'vertical_threat_index', 'defensive_range_index', 'goalkeeper_activity_index', 'crossing_value_index', 'box_delivery_index', 'set_piece_creator_index', 'dead_ball_threat_index', 'corner_threat_index', 'second_phase_corner_index', 'defensive_duel_index', 'high_press_defending_index', 'box_defender_index', 'defensive_security_index', 'disruption_index', 'passing_added_index', 'waltzing_all_round_index']:
    player_season[f'{col}_pctile'] = pct_rank(player_season[col].replace([np.inf, -np.inf], np.nan))
player_season['impact_score'] = player_season['impact_score_raw_pctile']

def role_label(row):
    vals = {
        'finisher': row.get('finisher_index_pctile', 0),
        'creator': row.get('creative_engine_index_pctile', 0),
        'progressor': row.get('progression_index_pctile', 0),
        'defender': row.get('ball_winner_index_pctile', 0),
        'two_way': (row.get('progression_index_pctile', 0) + row.get('defensive_index_pctile', 0)) / 2,
    }
    best = max(vals, key=vals.get)
    return {
        'finisher': 'Finisher / Shot Getter',
        'creator': 'Creator / Chance Builder',
        'progressor': 'Progressor / Territory Mover',
        'defender': 'Ball-Winner / Stopper',
        'two_way': 'Two-Way Connector',
    }[best]

player_season['role'] = player_season.apply(role_label, axis=1)
player_season['archetype'] = pd.cut(
    player_season['impact_score'],
    bins=[-1, 35, 65, 85, 101],
    labels=['Role Player', 'Useful Contributor', 'High-Impact Player', 'Elite Impact'],
).astype(str)

team_metric_cols = [c for c in events_df.columns if c in metric_cols]
team_match_actions = events_df.groupby(['season', 'match_file', 'date', 'match', 'team_id', 'team'], as_index=False)[team_metric_cols].sum()
team_minutes = minutes_df.groupby(['season', 'match_file', 'date', 'match', 'team_id', 'team'], as_index=False).agg(
    player_minutes=('minutes', 'sum'), players_used=('player_id', 'nunique'), gf_on=('gf_on', 'max'), ga_on=('ga_on', 'max')
)
team_match = team_match_actions.merge(team_minutes, on=['season', 'match_file', 'date', 'match', 'team_id', 'team'], how='left')
team_match['gd'] = team_match['gf_on'].fillna(0) - team_match['ga_on'].fillna(0)
team_match['team_action_value'] = (
    team_match.get('xg', 0) + team_match.get('xt_positive', 0) + team_match.get('carry_xt_positive', 0) +
    team_match.get('def_actions', 0) * 0.006 - team_match.get('losses', 0) * 0.004
)
team_match['team_threat'] = team_match.get('danger', 0) + team_match.get('xt_positive', 0) + team_match.get('xg', 0)
team_match['team_gda'] = team_match['gd'] + team_match['team_action_value']
team_match['field_tilt'] = team_match.get('field_tilt_touch', 0) / team_match.get('touches', 0).replace(0, np.nan)
team_match['territory_balance'] = team_match.get('territory_added', 0) / team_match.get('actions', 0).replace(0, np.nan)
team_match['pass_pct'] = team_match.get('completed_passes', 0) / team_match.get('passes', 0).replace(0, np.nan)
team_match['box_pressure'] = team_match.get('box_touches', 0) + team_match.get('box_entries', 0) * 2 + team_match.get('shots', 0) * 3
team_match['rest_defence_proxy'] = team_match.get('pressure_regains', 0) / team_match.get('losses', 0).replace(0, np.nan)
team_match['transition_threat'] = team_match.get('counter_threat_actions', 0) + team_match.get('progressive_carries', 0) * 0.5
team_match['team_touch_share'] = team_match.get('touches', 0) / team_match.groupby('match_file')['touches'].transform('sum').replace(0, np.nan)

team_keys = ['season', 'team']
team_season = team_match.groupby(team_keys, as_index=False).sum(numeric_only=True)
team_season['matches'] = team_match.groupby(team_keys)['match_file'].nunique().values
team_season['avg_touch_share'] = team_match.groupby(team_keys)['team_touch_share'].mean().values
team_ids = team_match.groupby(team_keys)['team_id'].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0]).reset_index()
team_season = team_season.merge(team_ids, on=team_keys, how='left')
for df in [team_match, team_season]:
    df['team_gda_pm'] = df['team_gda'] / df.get('matches', 1) if 'matches' in df else df['team_gda']
    df['team_threat_pm'] = df['team_threat'] / df.get('matches', 1) if 'matches' in df else df['team_threat']
    df['xg_per_shot'] = df.get('xg', 0) / df.get('shots', 0).replace(0, np.nan)
    df['xt_per100_actions'] = df.get('xt', 0) / df.get('actions', 0).replace(0, np.nan) * 100
    df['danger_per100_actions'] = df.get('danger', 0) / df.get('actions', 0).replace(0, np.nan) * 100
    df['ppda_proxy'] = df.get('passes', 0) / df.get('def_actions', 0).replace(0, np.nan)
    df['box_pressure_pm'] = df.get('box_pressure', 0) / df.get('matches', 1) if 'matches' in df else df.get('box_pressure', 0)
    df['field_tilt'] = df.get('field_tilt_touch', 0) / df.get('touches', 0).replace(0, np.nan)
    df['territory_balance'] = df.get('territory_added', 0) / df.get('actions', 0).replace(0, np.nan)
    df['pass_pct'] = df.get('completed_passes', 0) / df.get('passes', 0).replace(0, np.nan)
    df['box_pressure'] = df.get('box_touches', 0) + df.get('box_entries', 0) * 2 + df.get('shots', 0) * 3
    df['rest_defence_proxy'] = df.get('pressure_regains', 0) / df.get('losses', 0).replace(0, np.nan)
    df['transition_threat'] = df.get('counter_threat_actions', 0) + df.get('progressive_carries', 0) * 0.5
    df['security_pct'] = 1 - df.get('losses', 0) / df.get('actions', 0).replace(0, np.nan)
    df['cross_accuracy'] = df.get('accurate_crosses', 0) / df.get('crosses', 0).replace(0, np.nan)
    df['cross_threat_per_cross'] = df.get('cross_threat', 0) / df.get('crosses', 0).replace(0, np.nan)
    df['cross_xt_per_cross'] = df.get('cross_xt', 0) / df.get('crosses', 0).replace(0, np.nan)
    df['set_piece_pass_pct'] = df.get('completed_set_piece_passes', 0) / df.get('set_piece_passes', 0).replace(0, np.nan)
    df['set_piece_threat_per_action'] = (df.get('set_piece_xg', 0) + df.get('set_piece_xt', 0)) / df.get('set_piece_actions', 0).replace(0, np.nan)
    df['defensive_duel_win_pct'] = df.get('defensive_duels_won', 0) / df.get('defensive_duels', 0).replace(0, np.nan)
    df['ground_duel_win_pct'] = df.get('ground_duels_won', 0) / df.get('ground_duels', 0).replace(0, np.nan)
    df['xpass_completion_pct'] = df.get('xpass', 0) / df.get('passes', 0).replace(0, np.nan)
    df['pass_completion_above_expected'] = df.get('completed_passes', 0) - df.get('xpass', 0)
    df['pass_completion_oe_per_pass'] = df.get('pass_completion_oe', 0) / df.get('passes', 0).replace(0, np.nan)
    df['xdisruption_per_def_action'] = df.get('xdisruption', 0) / df.get('def_actions', 0).replace(0, np.nan)
    df['corner_xg_per_corner'] = df.get('corner_xg', 0) / df.get('corners', 0).replace(0, np.nan)
    df['corner_goal_rate'] = df.get('corner_goals', 0) / df.get('corners', 0).replace(0, np.nan)
    df['corner_second_phase_xg_share'] = df.get('corner_second_phase_xg', 0) / df.get('corner_xg', 0).replace(0, np.nan)
    df['crossing_value_index'] = df.get('cross_threat', 0) / df.get('matches', 1) * 2.0 + df.get('accurate_crosses', 0) / df.get('matches', 1) * 0.35 + df.get('cutbacks', 0) / df.get('matches', 1) * 0.6 + df['cross_accuracy'].fillna(0) * 2.0
    df['box_delivery_index'] = df.get('crosses_into_box', 0) / df.get('matches', 1) * 0.45 + df.get('six_yard_crosses', 0) / df.get('matches', 1) * 0.7 + df.get('cutbacks', 0) / df.get('matches', 1) * 0.65 + df.get('box_entries', 0) / df.get('matches', 1) * 0.25
    df['set_piece_creator_index'] = df.get('set_piece_xt', 0) / df.get('matches', 1) * 3.0 + df.get('set_piece_xg', 0) / df.get('matches', 1) * 4.0 + df.get('set_piece_box_entries', 0) / df.get('matches', 1) * 0.35 + df.get('set_piece_crosses', 0) / df.get('matches', 1) * 0.25
    df['dead_ball_threat_index'] = df.get('corners', 0) / df.get('matches', 1) * 0.15 + df.get('free_kicks', 0) / df.get('matches', 1) * 0.2 + df['set_piece_threat_per_action'].fillna(0) * 8.0
    df['corner_threat_index'] = df.get('corner_xg', 0) / df.get('matches', 1) * 5.0 + df.get('corner_second_phase_xg', 0) / df.get('matches', 1) * 6.0 + df.get('corner_shots', 0) / df.get('matches', 1) * 0.35 + df['corner_goal_rate'].fillna(0) * 2.0
    df['second_phase_corner_index'] = df.get('corner_second_phase_actions', 0) / df.get('matches', 1) * 0.25 + df.get('corner_second_phase_xg', 0) / df.get('matches', 1) * 8.0 + df['corner_second_phase_xg_share'].fillna(0) * 2.0
    df['disruption_index'] = df.get('xdisruption', 0) / df.get('matches', 1) * 3.0 + df['xdisruption_per_def_action'].fillna(0) * 8.0
    df['passing_added_index'] = df['pass_completion_oe_per_pass'].fillna(0) * 20.0 + df.get('pass_value_over_expected', 0) / df.get('matches', 1) * 5.0
    df['control_index'] = df['field_tilt'].fillna(0) * 40 + df['pass_pct'].fillna(0) * 25 + df['security_pct'].fillna(0) * 20 + df['territory_balance'].fillna(0) * 15

for team_name, idx in player_season.groupby('team').groups.items():
    team_gda = player_season.loc[idx, 'gda'].sum()
    team_threat = player_season.loc[idx, 'threat'].sum()
    if team_gda:
        player_season.loc[idx, 'gda_share_of_team'] = player_season.loc[idx, 'gda'] / team_gda
    if team_threat:
        player_season.loc[idx, 'threat_share_of_team'] = player_season.loc[idx, 'threat'] / team_threat

team_context = team_season.set_index('team')
league_gd_p90 = player_season['gd_on'].sum() / player_season['minutes'].replace(0, np.nan).sum() * 90
player_season['team_gd_p90_context'] = player_season['team'].map((team_context['gd'] / team_context['player_minutes'].replace(0, np.nan) * 90).to_dict()).fillna(league_gd_p90)
player_season['rapm_raw_p90'] = (
    (player_season['gd_on_p90'].fillna(0) - player_season['team_gd_p90_context'].fillna(0)) * 0.55
    + player_season['action_gda_p90'].fillna(0) * 0.25
    + player_season['xdisruption_p90'].fillna(0) * 0.10
    + player_season['pass_value_over_expected_p90'].fillna(0) * 0.10
)
player_season['rapm_regularized_p90'] = player_season['rapm_raw_p90'] * player_season['reliability'].fillna(0)
player_season['rapm_attack_p90'] = (
    player_season['threat_p90'].fillna(0) * 0.40
    + player_season['xt_p90'].fillna(0) * 0.25
    + player_season['xg_p90'].fillna(0) * 0.20
    + player_season['pass_value_over_expected_p90'].fillna(0) * 0.15
) * player_season['reliability'].fillna(0)
player_season['rapm_defense_p90'] = (
    player_season['xdisruption_p90'].fillna(0) * 0.50
    + player_season['defensive_index'].fillna(0) * 0.20
    + player_season['defensive_security_index'].fillna(0) * 0.20
    + player_season['rest_defence_index'].fillna(0) * 0.10
) * player_season['reliability'].fillna(0)
player_season['rapm_total'] = player_season['rapm_regularized_p90']
player_season['rapm_percentile'] = pct_rank(player_season['rapm_regularized_p90'].replace([np.inf, -np.inf], np.nan))

team_touch_share_map = team_season.set_index('team')['avg_touch_share'].to_dict()
player_season['team_touch_share'] = player_season['team'].map(team_touch_share_map).fillna(0.5)
player_season['opponent_touch_share'] = (1 - player_season['team_touch_share']).clip(lower=0.25, upper=0.75)
player_season['padj_interceptions'] = player_season['interceptions_p90'] * (0.5 / player_season['opponent_touch_share'])
player_season['padj_tackles'] = player_season['tackles_p90'] * (0.5 / player_season['opponent_touch_share'])
player_season['duels'] = player_season.get('defensive_duels', 0) + player_season.get('offensive_duels', 0) + player_season.get('aerials', 0)
player_season['duels_p90'] = player_season['duels'] / player_season['minutes'].replace(0, np.nan) * 90
player_season['duels_won'] = player_season.get('defensive_duels_won', 0) + player_season.get('offensive_duels_won', 0) + player_season.get('aerials_won', 0)
player_season['duel_win_pct'] = player_season['duels_won'] / player_season['duels'].replace(0, np.nan)
player_season['goal_conversion_pct'] = player_season['goals'] / player_season['shots'].replace(0, np.nan)
player_season['penalty_conversion_pct'] = player_season['penalty_goals'] / player_season['penalty_shots'].replace(0, np.nan)

core_player_cols = [
    'season', 'team', 'player_id', 'player', 'position_group', 'matches', 'minutes', 'role', 'archetype',
    'impact_score', 'gda', 'gda_p90', 'gda_reliable_p90', 'gd_on', 'gf_on', 'ga_on',
    'action_gda', 'action_gda_p90', 'threat', 'threat_p90', 'xg', 'xg_p90', 'shots', 'shots_p90',
    'goals', 'goals_p90', 'goals_minus_xg', 'on_target_pct', 'xt', 'xt_p90', 'xt_per100_actions',
    'passes', 'completed_passes', 'pass_pct', 'progressive_passes', 'progressive_passes_p90',
    'final3_entries', 'box_entries', 'carries', 'progressive_carries', 'def_actions', 'def_actions_p90',
    'recoveries', 'tackles', 'interceptions', 'clearances', 'pressure_regains', 'losses', 'security_pct',
    'attacking_index', 'progression_index', 'defensive_index', 'creative_engine_index', 'ball_winner_index',
    'finisher_index', 'chance_quality_index', 'needle_mover_index', 'chaos_creator_index',
    'press_resistance_index', 'rest_defence_index', 'territory_domination_index', 'connector_index',
    'directness_index', 'vertical_threat_index', 'defensive_range_index', 'goalkeeper_activity_index',
    'crossing_value_index', 'box_delivery_index', 'set_piece_creator_index', 'dead_ball_threat_index',
    'corner_threat_index', 'second_phase_corner_index', 'xpass_completion_pct',
    'pass_completion_above_expected', 'pass_value_over_expected', 'xdisruption', 'xdisruption_p90',
    'disruption_index', 'passing_added_index', 'rapm_regularized_p90', 'rapm_attack_p90',
    'rapm_defense_p90', 'rapm_percentile',
    'defensive_duel_index', 'high_press_defending_index', 'box_defender_index', 'defensive_security_index',
    'waltzing_all_round_index', 'box_presence_index', 'transition_index', 'gda_share_of_team', 'threat_share_of_team'
]
core_player_cols = [c for c in core_player_cols if c in player_season.columns]

core_team_cols = [
    'season', 'team', 'matches', 'gf_on', 'ga_on', 'gd', 'team_gda', 'team_gda_pm', 'team_threat',
    'team_threat_pm', 'xg', 'shots', 'xg_per_shot', 'xt', 'xt_per100_actions', 'danger',
    'danger_per100_actions', 'passes', 'completed_passes', 'pass_pct', 'field_tilt', 'territory_balance',
    'box_pressure', 'box_pressure_pm', 'rest_defence_proxy', 'transition_threat', 'ppda_proxy',
    'security_pct', 'control_index', 'players_used'
]
core_team_cols = [c for c in core_team_cols if c in team_season.columns]

action_values = events_df.copy()
zone_summary = events_df.groupby(['zone'], as_index=False).agg(
    actions=('actions', 'sum'), xg=('xg', 'sum'), xt=('xt', 'sum'), danger=('danger', 'sum'),
    touches=('touches', 'sum'), shots=('shots', 'sum'), goals=('goals', 'sum')
).sort_values('danger', ascending=False)

metric_dictionary = []
for col in sorted(set(player_season.columns).union(team_season.columns).union(player_match.columns).union(team_match.columns)):
    if col in {'season', 'date', 'match', 'match_file', 'team_id', 'team', 'player_id', 'player', 'role', 'archetype', 'position_group', 'shirt_no'}:
        family = 'identity'
    elif any(k in col for k in ['rapm']):
        family = 'rapm'
    elif any(k in col for k in ['xpass', 'pass_completion_oe', 'pass_value_over_expected', 'passing_added']):
        family = 'xpass'
    elif any(k in col for k in ['xdisruption', 'disruption']):
        family = 'xdisruption'
    elif any(k in col for k in ['corner', 'set_piece', 'dead_ball']):
        family = 'set_pieces'
    elif any(k in col for k in ['gda', 'gd_on', 'gf_on', 'ga_on']):
        family = 'goal_difference_added'
    elif any(k in col for k in ['xg', 'shot', 'goal', 'finish', 'psxg']):
        family = 'shooting'
    elif any(k in col for k in ['xt', 'progress', 'territory', 'entry', 'carry']):
        family = 'progression'
    elif any(k in col for k in ['pass', 'control', 'security', 'loss']):
        family = 'possession'
    elif any(k in col for k in ['def', 'tackle', 'interception', 'clearance', 'recover', 'pressure', 'aerial']):
        family = 'defending'
    elif any(k in col for k in ['index', 'score', 'pctile', 'role', 'archetype']):
        family = 'model_index'
    else:
        family = 'other'
    metric_dictionary.append({'metric': col, 'family': family, 'description': col.replace('_', ' ')})

outputs = {
    'events_enriched.csv': action_values,
    'player_match_metrics.csv': player_match,
    'player_season_metrics.csv': player_season,
    'player_season_core.csv': player_season[core_player_cols].sort_values('impact_score', ascending=False),
    'team_match_metrics.csv': team_match,
    'team_season_metrics.csv': team_season,
    'team_season_core.csv': team_season[core_team_cols].sort_values('team_gda_pm', ascending=False),
    'match_metrics.csv': matches_df,
    'zone_summary.csv': zone_summary,
    'metric_dictionary.csv': pd.DataFrame(metric_dictionary),
}

for name, df in outputs.items():
    df = df.replace([np.inf, -np.inf], np.nan)
    df.to_csv(OUT_DIR / name, index=False)

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

ID = ['season', 'team', 'player_id', 'player', 'position_group', 'matches', 'minutes']
PLAYER_STANDARD = ID + [
    'role', 'archetype', 'impact_score', 'started', 'gf_on', 'ga_on', 'gd_on', 'gd_on_p90',
    'actions', 'actions_p90', 'touches', 'touches_p90', 'security_pct', 'availability_score', 'reliability'
]
PLAYER_SHOOTING = ID + [
    'goals', 'goals_p90', 'shots', 'shots_p90', 'on_target_shots', 'on_target_pct', 'xg', 'xg_p90',
    'xg_per_shot', 'goals_minus_xg', 'finishing_ratio', 'psxg_proxy', 'psxg_proxy_p90',
    'xg_per_box_touch', 'chance_quality_index', 'volume_quality_index', 'finisher_index'
]
PLAYER_PASSING = ID + [
    'passes', 'passes_p90', 'completed_passes', 'completed_passes_p90', 'pass_pct',
    'xpass', 'xpass_p90', 'xpass_completion_pct', 'pass_completion_above_expected',
    'pass_completion_oe', 'pass_completion_oe_per_pass', 'pass_value_over_expected',
    'pass_value_over_expected_p90', 'passing_added_index',
    'forward_passes', 'forward_pass_pct', 'backward_passes', 'lateral_passes', 'long_passes',
    'progressive_passes', 'progressive_passes_p90', 'progressive_pass_pct',
    'progressive_pass_per_complete', 'final3_entries', 'final3_entries_p90',
    'box_entries', 'box_entries_p90', 'box_entry_rate_from_final3', 'connector_index', 'creative_engine_index'
]
PLAYER_CROSSING = ID + [
    'crosses', 'crosses_p90', 'accurate_crosses', 'accurate_crosses_p90', 'cross_accuracy',
    'open_play_crosses', 'open_play_crosses_p90', 'set_piece_crosses', 'set_piece_crosses_p90',
    'corner_crosses', 'corner_crosses_p90', 'free_kick_crosses', 'free_kick_crosses_p90',
    'early_crosses', 'early_crosses_p90', 'deep_crosses', 'deep_crosses_p90',
    'cutbacks', 'cutbacks_p90', 'cutback_share', 'six_yard_crosses', 'six_yard_crosses_p90',
    'crosses_into_box', 'crosses_into_box_p90', 'cross_xt', 'cross_xt_p90',
    'cross_xt_per_cross', 'cross_threat', 'cross_threat_p90', 'cross_threat_per_cross',
    'crossing_value_index', 'box_delivery_index'
]
PLAYER_POSSESSION = ID + [
    'touches', 'touches_p90', 'box_touches', 'box_touches_p90', 'box_touch_share',
    'final3_touches', 'final3_touch_share', 'central_touch_share', 'wide_touch_share',
    'halfspace_touches_p90', 'take_ons', 'take_ons_p90', 'successful_take_ons',
    'take_on_pct', 'losses', 'losses_p90', 'security_pct', 'dispossessed',
    'press_resistance_index', 'chaos_creator_index'
]
PLAYER_PROGRESSION = ID + [
    'xt', 'xt_p90', 'xt_positive', 'xt_negative', 'xt_risk_reward', 'xt_per_touch',
    'xt_per100_actions', 'xt_per100_passes', 'carries', 'carries_p90', 'carry_distance',
    'progressive_carries', 'progressive_carries_p90', 'carry_progression_share',
    'pass_progression_share', 'territory_added', 'territory_added_p90', 'territory_per_action',
    'territory_domination_index', 'directness_index', 'vertical_threat_index', 'wide_progression_index'
]
PLAYER_DEFENDING = ID + [
    'def_actions', 'def_actions_p90', 'def_action_share', 'tackles', 'tackles_p90',
    'xdisruption', 'xdisruption_p90', 'xdisruption_per_def_action', 'disruption_index',
    'tackles_won', 'tackle_win_pct', 'interceptions', 'interceptions_p90', 'recoveries',
    'recoveries_p90', 'clearances', 'clearances_p90', 'aerials', 'aerials_p90',
    'aerials_won', 'aerial_win_pct', 'blocks', 'pressure_regains', 'pressure_regains_p90',
    'high_regains', 'high_regains_p90', 'counterpress_regains', 'counterpress_regains_p90',
    'defensive_duels', 'defensive_duels_p90', 'defensive_duels_won', 'defensive_duel_win_pct',
    'ground_duels', 'ground_duels_p90', 'ground_duels_won', 'ground_duel_win_pct',
    'defensive_third_def_actions', 'defensive_third_def_actions_p90', 'middle_third_def_actions',
    'middle_third_def_actions_p90', 'final_third_def_actions', 'final_third_def_actions_p90',
    'last_line_actions', 'last_line_actions_p90', 'box_def_actions', 'box_def_actions_p90',
    'def_third_action_share', 'mid_third_def_action_share', 'high_def_action_share',
    'regain_to_threat_rate', 'rest_defence_index', 'ball_winner_index', 'defensive_range_index',
    'defensive_duel_index', 'high_press_defending_index', 'box_defender_index', 'defensive_security_index'
]
PLAYER_SET_PIECES = ID + [
    'set_piece_actions', 'set_piece_actions_p90', 'set_piece_passes', 'set_piece_passes_p90',
    'completed_set_piece_passes', 'completed_set_piece_passes_p90', 'set_piece_pass_pct',
    'corners', 'corners_p90', 'free_kicks', 'free_kicks_p90', 'throw_ins', 'throw_ins_p90',
    'set_piece_crosses', 'set_piece_crosses_p90', 'corner_crosses', 'corner_crosses_p90',
    'free_kick_crosses', 'free_kick_crosses_p90', 'set_piece_box_entries', 'set_piece_box_entries_p90',
    'set_piece_shots', 'set_piece_shots_p90', 'set_piece_xg', 'set_piece_xg_p90',
    'set_piece_goals', 'set_piece_goals_p90', 'set_piece_xt', 'set_piece_xt_p90',
    'set_piece_threat_per_action', 'set_piece_creator_index', 'dead_ball_threat_index',
    'corner_phase_actions', 'corner_phase_actions_p90', 'corner_first_phase_actions',
    'corner_first_phase_actions_p90', 'corner_second_phase_actions', 'corner_second_phase_actions_p90',
    'corner_shots', 'corner_shots_p90', 'corner_xg', 'corner_xg_p90', 'corner_xg_per_corner',
    'corner_goals', 'corner_goal_rate', 'corner_first_phase_shots', 'corner_first_phase_xg',
    'corner_first_phase_goals', 'corner_second_phase_shots', 'corner_second_phase_xg',
    'corner_second_phase_xg_p90', 'corner_second_phase_goals', 'corner_second_phase_xg_share',
    'corner_threat_index', 'second_phase_corner_index'
]
PLAYER_GOALKEEPING = ID + [
    'saves', 'saves_p90', 'claims', 'claims_p90', 'keeper_pickups', 'keeper_pickups_p90',
    'smothers', 'smothers_p90', 'keeper_sweeper_actions', 'keeper_sweeper_actions_p90',
    'punches', 'punches_p90', 'penalties_faced', 'penalties_faced_p90',
    'gk_actions', 'gk_actions_p90', 'goalkeeper_activity_index'
]
PLAYER_GDA = ID + [
    'gda', 'gda_p90', 'gda_reliable_p90', 'action_gda', 'action_gda_p90',
    'gd_on', 'gd_on_p90', 'gf_on', 'ga_on', 'gda_share_of_team',
    'needle_mover_index', 'waltzing_all_round_index'
]
PLAYER_DANGER = ID + [
    'threat', 'threat_p90', 'threat_reliable_p90', 'danger', 'danger_p90',
    'danger_per_touch', 'danger_per_action', 'chain_xt', 'chain_xt_p90',
    'threat_per_touch', 'threat_share_of_team', 'high_value_touches',
    'high_value_touches_p90', 'counter_threat_actions', 'counter_threat_actions_p90',
    'transition_index'
]
PLAYER_IMPACT = ID + [
    'role', 'archetype', 'impact_score', 'impact_score_raw', 'waltzing_all_round_index',
    'attacking_index', 'progression_index', 'defensive_index', 'security_index',
    'box_presence_index', 'transition_index', 'creative_engine_index', 'ball_winner_index',
    'finisher_index', 'chance_quality_index', 'volume_quality_index', 'needle_mover_index',
    'chaos_creator_index', 'press_resistance_index', 'rest_defence_index', 'territory_domination_index',
    'connector_index', 'halfspace_value_index', 'directness_index', 'vertical_threat_index',
    'defensive_range_index', 'goalkeeper_activity_index'
]
PLAYER_XPASS = ID + [
    'passes', 'completed_passes', 'pass_pct', 'xpass', 'xpass_p90', 'xpass_completion_pct',
    'pass_completion_above_expected', 'pass_completion_oe', 'pass_completion_oe_p90',
    'pass_completion_oe_per_pass', 'pass_value_over_expected', 'pass_value_over_expected_p90',
    'progressive_passes', 'box_entries', 'final3_entries', 'passing_added_index'
]
PLAYER_XDISRUPTION = ID + [
    'def_actions', 'def_actions_p90', 'xdisruption', 'xdisruption_p90',
    'xdisruption_per_def_action', 'disruption_index', 'high_regains', 'counterpress_regains',
    'pressure_regains', 'defensive_duels', 'defensive_duel_win_pct',
    'last_line_actions', 'box_def_actions', 'defensive_security_index'
]
PLAYER_RAPM = ID + [
    'rapm_regularized_p90', 'rapm_raw_p90', 'rapm_total', 'rapm_percentile',
    'rapm_attack_p90', 'rapm_defense_p90', 'team_gd_p90_context',
    'gd_on_p90', 'action_gda_p90', 'xdisruption_p90', 'pass_value_over_expected_p90',
    'gda_p90', 'threat_p90', 'xt_p90', 'impact_score', 'role', 'archetype'
]

TEAM_STANDARD = ['season', 'team', 'matches', 'gf_on', 'ga_on', 'gd', 'team_gda', 'team_gda_pm', 'team_threat', 'team_threat_pm', 'players_used']
TEAM_ATTACK = ['season', 'team', 'matches', 'team_threat_pm', 'goals', 'shots', 'xg', 'xg_per_shot', 'danger', 'danger_per100_actions', 'box_pressure', 'box_pressure_pm', 'transition_threat']
TEAM_PASSING = ['season', 'team', 'matches', 'passes', 'completed_passes', 'pass_pct', 'progressive_passes', 'final3_entries', 'box_entries', 'xt', 'xt_per100_actions']
TEAM_CROSSING = ['season', 'team', 'matches', 'crosses', 'accurate_crosses', 'cross_accuracy', 'open_play_crosses', 'set_piece_crosses', 'cutbacks', 'six_yard_crosses', 'crosses_into_box', 'cross_xt', 'cross_threat', 'crossing_value_index', 'box_delivery_index']
TEAM_SET_PIECES = ['season', 'team', 'matches', 'set_piece_actions', 'set_piece_passes', 'set_piece_pass_pct', 'corners', 'free_kicks', 'throw_ins', 'set_piece_crosses', 'set_piece_box_entries', 'set_piece_shots', 'set_piece_xg', 'set_piece_goals', 'set_piece_xt', 'set_piece_threat_per_action', 'corner_shots', 'corner_xg', 'corner_xg_per_corner', 'corner_goals', 'corner_goal_rate', 'corner_first_phase_xg', 'corner_second_phase_xg', 'corner_second_phase_xg_share', 'set_piece_creator_index', 'dead_ball_threat_index', 'corner_threat_index', 'second_phase_corner_index']
TEAM_DEFENDING = ['season', 'team', 'matches', 'def_actions', 'tackles', 'interceptions', 'recoveries', 'clearances', 'pressure_regains', 'high_regains', 'counterpress_regains', 'defensive_duels', 'defensive_duel_win_pct', 'ground_duel_win_pct', 'last_line_actions', 'box_def_actions', 'rest_defence_proxy', 'ppda_proxy']
TEAM_POSSESSION = ['season', 'team', 'matches', 'touches', 'field_tilt', 'territory_balance', 'security_pct', 'control_index', 'losses']

def ws_col(df, col, default=np.nan):
    return df[col] if col in df.columns else default

wyscout_style = pd.DataFrame({
    'Player': player_season['player'],
    'Team': player_season['team'],
    'Position': ws_col(player_season, 'position_group', ''),
    'Matches played': player_season['matches'],
    'Minutes played': player_season['minutes'],
    'Goals': ws_col(player_season, 'goals', 0),
    'xG': ws_col(player_season, 'xg', 0),
    'Assists': ws_col(player_season, 'assists', 0),
    'xA': ws_col(player_season, 'xa', 0),
    'Duels per 90': ws_col(player_season, 'duels_p90', np.nan),
    'Duels won, %': ws_col(player_season, 'duel_win_pct', np.nan),
    'Successful defensive actions per 90': ws_col(player_season, 'def_actions_p90', np.nan),
    'Defensive duels per 90': ws_col(player_season, 'defensive_duels_p90', np.nan),
    'Defensive duels won, %': ws_col(player_season, 'defensive_duel_win_pct', np.nan),
    'Aerial duels per 90': ws_col(player_season, 'aerials_p90', np.nan),
    'Aerial duels won, %': ws_col(player_season, 'aerial_win_pct', np.nan),
    'Sliding tackles per 90': ws_col(player_season, 'tackles_p90', np.nan),
    'PAdj Sliding tackles': ws_col(player_season, 'padj_tackles', np.nan),
    'Shots blocked per 90': ws_col(player_season, 'blocks_p90', np.nan),
    'Interceptions per 90': ws_col(player_season, 'interceptions_p90', np.nan),
    'PAdj Interceptions': ws_col(player_season, 'padj_interceptions', np.nan),
    'Fouls per 90': ws_col(player_season, 'fouls_p90', np.nan),
    'Successful attacking actions per 90': ws_col(player_season, 'successful_attacking_actions_p90', np.nan),
    'Goals per 90': ws_col(player_season, 'goals_p90', np.nan),
    'Non-penalty goals': ws_col(player_season, 'non_penalty_goals', 0),
    'Non-penalty goals per 90': ws_col(player_season, 'non_penalty_goals_p90', np.nan),
    'xG per 90': ws_col(player_season, 'xg_p90', np.nan),
    'Head goals': ws_col(player_season, 'head_goals', 0),
    'Head goals per 90': ws_col(player_season, 'head_goals_p90', np.nan),
    'Shots': ws_col(player_season, 'shots', 0),
    'Shots per 90': ws_col(player_season, 'shots_p90', np.nan),
    'Shots on target, %': ws_col(player_season, 'on_target_pct', np.nan),
    'Goal conversion, %': ws_col(player_season, 'goal_conversion_pct', np.nan),
    'Assists per 90': ws_col(player_season, 'assists_p90', np.nan),
    'Crosses per 90': ws_col(player_season, 'crosses_p90', np.nan),
    'Accurate crosses, %': ws_col(player_season, 'cross_accuracy', np.nan),
    'Crosses from left flank per 90': ws_col(player_season, 'left_flank_crosses_p90', np.nan),
    'Accurate crosses from left flank, %': ws_col(player_season, 'accurate_left_flank_cross_pct', np.nan),
    'Crosses from right flank per 90': ws_col(player_season, 'right_flank_crosses_p90', np.nan),
    'Accurate crosses from right flank, %': ws_col(player_season, 'accurate_right_flank_cross_pct', np.nan),
    'Crosses to goalie box per 90': ws_col(player_season, 'crosses_to_goalie_box_p90', np.nan),
    'Dribbles per 90': ws_col(player_season, 'take_ons_p90', np.nan),
    'Successful dribbles, %': ws_col(player_season, 'take_on_pct', np.nan),
    'Offensive duels per 90': ws_col(player_season, 'offensive_duels_p90', np.nan),
    'Offensive duels won, %': ws_col(player_season, 'offensive_duel_win_pct', np.nan),
    'Touches in box per 90': ws_col(player_season, 'box_touches_p90', np.nan),
    'Progressive runs per 90': ws_col(player_season, 'progressive_runs_p90', np.nan),
    'Accelerations per 90': ws_col(player_season, 'accelerations_p90', np.nan),
    'Received passes per 90': ws_col(player_season, 'received_passes_p90', np.nan),
    'Received long passes per 90': ws_col(player_season, 'received_long_passes_p90', np.nan),
    'Fouls suffered per 90': ws_col(player_season, 'fouled_p90', np.nan),
    'Passes per 90': ws_col(player_season, 'passes_p90', np.nan),
    'Accurate passes, %': ws_col(player_season, 'pass_pct', np.nan),
    'Forward passes per 90': ws_col(player_season, 'forward_passes_p90', np.nan),
    'Accurate forward passes, %': ws_col(player_season, 'accurate_forward_pass_pct', np.nan),
    'Back passes per 90': ws_col(player_season, 'backward_passes_p90', np.nan),
    'Accurate back passes, %': ws_col(player_season, 'accurate_back_pass_pct', np.nan),
    'Lateral passes per 90': ws_col(player_season, 'lateral_passes_p90', np.nan),
    'Accurate lateral passes, %': ws_col(player_season, 'accurate_lateral_pass_pct', np.nan),
    'Short / medium passes per 90': ws_col(player_season, 'short_medium_passes_p90', np.nan),
    'Accurate short / medium passes, %': ws_col(player_season, 'accurate_short_medium_pass_pct', np.nan),
    'Long passes per 90': ws_col(player_season, 'long_passes_p90', np.nan),
    'Accurate long passes, %': ws_col(player_season, 'accurate_long_pass_pct', np.nan),
    'Average pass length, m': ws_col(player_season, 'avg_pass_length', np.nan),
    'Average long pass length, m': ws_col(player_season, 'avg_long_pass_length', np.nan),
    'xA per 90': ws_col(player_season, 'xa_p90', np.nan),
    'Shot assists per 90': ws_col(player_season, 'shot_assists_p90', np.nan),
    'Second assists per 90': ws_col(player_season, 'second_assists_p90', np.nan),
    'Third assists per 90': ws_col(player_season, 'third_assists_p90', np.nan),
    'Smart passes per 90': ws_col(player_season, 'smart_passes_p90', np.nan),
    'Accurate smart passes, %': ws_col(player_season, 'accurate_smart_pass_pct', np.nan),
    'Key passes per 90': ws_col(player_season, 'key_passes_p90', np.nan),
    'Passes to final third per 90': ws_col(player_season, 'passes_into_final3_p90', np.nan),
    'Passes to penalty area per 90': ws_col(player_season, 'passes_into_box_p90', np.nan),
    'Through passes per 90': ws_col(player_season, 'through_passes_p90', np.nan),
    'Accurate through passes, %': ws_col(player_season, 'accurate_through_pass_pct', np.nan),
    'Deep completions per 90': ws_col(player_season, 'deep_completions_p90', np.nan),
    'Deep completed crosses per 90': ws_col(player_season, 'deep_completed_crosses_p90', np.nan),
    'Progressive passes per 90': ws_col(player_season, 'progressive_passes_p90', np.nan),
    'Accurate progressive passes, %': ws_col(player_season, 'progressive_pass_pct', np.nan),
    'Conceded goals': ws_col(player_season, 'conceded_goals', np.nan),
    'Conceded goals per 90': ws_col(player_season, 'conceded_goals_p90', np.nan),
    'Shots against': ws_col(player_season, 'shots_against', np.nan),
    'Shots against per 90': ws_col(player_season, 'shots_against_p90', np.nan),
    'Clean sheets': ws_col(player_season, 'clean_sheets', np.nan),
    'Save rate, %': ws_col(player_season, 'save_rate', np.nan),
    'xG against': ws_col(player_season, 'xg_against', np.nan),
    'xG against per 90': ws_col(player_season, 'xg_against_p90', np.nan),
    'Prevented goals': ws_col(player_season, 'prevented_goals', np.nan),
    'Prevented goals per 90': ws_col(player_season, 'prevented_goals_p90', np.nan),
    'Free kicks per 90': ws_col(player_season, 'free_kicks_p90', np.nan),
    'Direct free kicks per 90': ws_col(player_season, 'direct_free_kicks_p90', np.nan),
    'Direct free kicks on target, %': ws_col(player_season, 'direct_free_kick_on_target_pct', np.nan),
    'Corners per 90': ws_col(player_season, 'corners_p90', np.nan),
    'Penalties taken': ws_col(player_season, 'penalty_shots', 0),
    'Penalty conversion, %': ws_col(player_season, 'penalty_conversion_pct', np.nan),
}).sort_values(['Minutes played', 'Player'], ascending=[False, True])

excel_sheets = {
    'Info': pd.DataFrame([
        {'item': 'source', 'value': str(SOURCE_DIR)},
        {'item': 'credit', 'value': f'Source: Opta/StatsPerform | Models by Marc Lamberts - Waltzing Analytics | {TODAY}'},
        {'item': 'matches_loaded', 'value': len(matches_df)},
        {'item': 'players', 'value': player_season['player_id'].nunique()},
        {'item': 'teams', 'value': team_season['team'].nunique()},
        {'item': 'note', 'value': 'xG, xT, danger and GDA are transparent event-data proxy models.'},
    ]),
    'Player Standard': player_season[existing_cols(player_season, PLAYER_STANDARD)].sort_values('impact_score', ascending=False),
    'Player Wyscout Style': wyscout_style,
    'Player Shooting': player_season[existing_cols(player_season, PLAYER_SHOOTING)].sort_values('xg_p90', ascending=False),
    'Player Passing': player_season[existing_cols(player_season, PLAYER_PASSING)].sort_values('creative_engine_index', ascending=False),
    'Player Crossing': player_season[existing_cols(player_season, PLAYER_CROSSING)].sort_values('crossing_value_index', ascending=False),
    'Player Possession': player_season[existing_cols(player_season, PLAYER_POSSESSION)].sort_values('press_resistance_index', ascending=False),
    'Player Progression': player_season[existing_cols(player_season, PLAYER_PROGRESSION)].sort_values('vertical_threat_index', ascending=False),
    'Player Defending': player_season[existing_cols(player_season, PLAYER_DEFENDING)].sort_values('ball_winner_index', ascending=False),
    'Player Goalkeeping': player_season[existing_cols(player_season, PLAYER_GOALKEEPING)].sort_values('goalkeeper_activity_index', ascending=False),
    'Player Set Pieces': player_season[existing_cols(player_season, PLAYER_SET_PIECES)].sort_values('set_piece_creator_index', ascending=False),
    'Player RAPM': player_season[existing_cols(player_season, PLAYER_RAPM)].sort_values('rapm_regularized_p90', ascending=False),
    'Player xPass': player_season[existing_cols(player_season, PLAYER_XPASS)].sort_values('passing_added_index', ascending=False),
    'Player xDisruption': player_season[existing_cols(player_season, PLAYER_XDISRUPTION)].sort_values('disruption_index', ascending=False),
    'Player GDA': player_season[existing_cols(player_season, PLAYER_GDA)].sort_values('gda_p90', ascending=False),
    'Player xT Danger': player_season[existing_cols(player_season, PLAYER_DANGER)].sort_values('threat_p90', ascending=False),
    'Player Impact': player_season[existing_cols(player_season, PLAYER_IMPACT)].sort_values('waltzing_all_round_index', ascending=False),
    'Team Standard': team_season[existing_cols(team_season, TEAM_STANDARD)].sort_values('team_gda_pm', ascending=False),
    'Team Attack': team_season[existing_cols(team_season, TEAM_ATTACK)].sort_values('team_threat_pm', ascending=False),
    'Team Passing': team_season[existing_cols(team_season, TEAM_PASSING)].sort_values('xt_per100_actions', ascending=False),
    'Team Crossing': team_season[existing_cols(team_season, TEAM_CROSSING)].sort_values('crossing_value_index', ascending=False),
    'Team Set Pieces': team_season[existing_cols(team_season, TEAM_SET_PIECES)].sort_values('set_piece_creator_index', ascending=False),
    'Team Possession': team_season[existing_cols(team_season, TEAM_POSSESSION)].sort_values('control_index', ascending=False),
    'Team Defending': team_season[existing_cols(team_season, TEAM_DEFENDING)].sort_values('rest_defence_proxy', ascending=False),
    'Metric Dictionary': pd.DataFrame(metric_dictionary),
}

DOWNLOAD_DIR = Path('/Users/marclamberts/Downloads/Waltzing Analytics')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_PATH = DOWNLOAD_DIR / f'{SEASON} - Season Data.xlsx'
try:
    with pd.ExcelWriter(EXCEL_PATH, engine='xlsxwriter') as writer:
        workbook = writer.book
        header_fmt = workbook.add_format({'bold': True, 'font_color': 'white', 'bg_color': '#1F4E78', 'border': 1})
        text_fmt = workbook.add_format({'num_format': '@'})
        num_fmt = workbook.add_format({'num_format': '#,##0.000'})
        int_fmt = workbook.add_format({'num_format': '#,##0'})
        pct_fmt = workbook.add_format({'num_format': '0.0%'})
        for sheet_name, df in excel_sheets.items():
            clean = df.replace([np.inf, -np.inf], np.nan).copy()
            clean.to_excel(writer, sheet_name=sheet_name[:31], index=False)
            ws = writer.sheets[sheet_name[:31]]
            ws.freeze_panes(1, 0)
            if len(clean.columns) > 0:
                ws.autofilter(0, 0, max(len(clean), 1), len(clean.columns) - 1)
            for col_idx, col in enumerate(clean.columns):
                series = clean[col].head(300).astype(str)
                width = min(28, max(9, len(str(col)) + 2, int(series.map(len).max() if len(series) else 9) + 1))
                fmt = text_fmt
                if pd.api.types.is_numeric_dtype(clean[col]):
                    if col.endswith('_pct') or col.endswith('_share') or col in {'pass_pct', 'security_pct', 'field_tilt', 'territory_balance', 'on_target_pct', 'take_on_pct', 'aerial_win_pct', 'tackle_win_pct'}:
                        fmt = pct_fmt
                    elif col in {'matches', 'actions', 'touches', 'passes', 'shots', 'goals', 'def_actions', 'saves'}:
                        fmt = int_fmt
                    else:
                        fmt = num_fmt
                ws.set_column(col_idx, col_idx, width, fmt)
            for col_idx, col in enumerate(clean.columns):
                ws.write(0, col_idx, col, header_fmt)
except Exception:
    with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
        for sheet_name, df in excel_sheets.items():
            df.replace([np.inf, -np.inf], np.nan).to_excel(writer, sheet_name=sheet_name[:31], index=False)

meta = {
    'source': str(SOURCE_DIR),
    'output': str(OUT_DIR),
    'excel_workbook': str(EXCEL_PATH),
    'created': datetime.now().isoformat(timespec='seconds'),
    'credit': f'Source: Opta/StatsPerform | Models by Marc Lamberts - Waltzing Analytics | {TODAY}',
    'matches_loaded': int(len(matches_df)),
    'events_loaded': int(len(events_df)),
    'players': int(player_season['player_id'].nunique()),
    'teams': int(team_season['team_id'].nunique()),
    'notes': [
        'xG, xT, danger, and post-shot xG are transparent proxy models built from event coordinates and qualifiers.',
        'GDA combines goals while on pitch with action value. On-pitch intervals are estimated from lineup and substitution events.',
        'xPass estimates pass completion probability from distance, direction, end zone, cross and set-piece context.',
        'xDisruption estimates defensive possession disruption value from location, action type, success and regain context.',
        'RAPM is a regularized adjusted plus-minus style proxy using on-pitch GD, team context, action value, xDisruption and xPass value.',
        'Corner first phase is corner delivery plus next 12 seconds; second phase is 12-45 seconds after the corner for the same team.',
        'Notebook reads directly from /Users/marclamberts/Event data/Ecuador 2026.',
    ],
}
(OUT_DIR / 'metrics_meta.json').write_text(json.dumps(meta, indent=2))

print(f'Saved expanded Ecuador 2026 metric package to: {OUT_DIR}')
for name, df in outputs.items():
    print(f'  {name:<28} rows={len(df):>7} cols={len(df.columns):>4} size={(OUT_DIR / name).stat().st_size / 1024:>9.1f} KB')
print(f"  metrics_meta.json             size={(OUT_DIR / 'metrics_meta.json').stat().st_size / 1024:>9.1f} KB")
print(f"  {EXCEL_PATH.name:<28} size={EXCEL_PATH.stat().st_size / 1024:>9.1f} KB")

print('\nTop players by impact_score:')
preview_cols = [c for c in ['team', 'player', 'minutes', 'role', 'archetype', 'impact_score', 'gda_p90', 'threat_p90', 'xt_p90', 'xg_p90', 'def_actions_p90'] if c in player_season.columns]
print(player_season.sort_values('impact_score', ascending=False)[preview_cols].head(20).to_string(index=False))

print('\nTop teams by team_gda_pm:')
team_preview = [c for c in ['team', 'matches', 'team_gda_pm', 'team_threat_pm', 'xg_per_shot', 'field_tilt', 'control_index'] if c in team_season.columns]
print(team_season.sort_values('team_gda_pm', ascending=False)[team_preview].head(20).to_string(index=False))
